In [1]:
# Stage 3: observable-data learner for phantom interactions

%load_ext autoreload
%autoreload 2

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import expm
from scipy.optimize import least_squares

import common_function as cf


# ------------------------------------------------------------------
# Reproducibility and output directories
# ------------------------------------------------------------------

stage3_seed = 20260808
stage3_rng = np.random.default_rng(
    stage3_seed
)

tol = 1e-10

stage2_result_dir = Path(
    "results_stage2"
)

stage3_result_dir = Path(
    "results_stage3"
)

stage3_figure_dir = Path(
    "figures_stage3"
)

stage3_result_dir.mkdir(
    exist_ok=True
)

stage3_figure_dir.mkdir(
    exist_ok=True
)


np.set_printoptions(
    precision=6,
    suppress=True
)

In [2]:
# ------------------------------------------------------------------
# Load the frozen Stage 2 benchmark
# ------------------------------------------------------------------

benchmark_path = (
    stage2_result_dir
    / "stage3_learner_benchmark.npz"
)


assert benchmark_path.exists(), (
    f"Benchmark file not found: {benchmark_path.resolve()}"
)


benchmark_data = np.load(
    benchmark_path,
    allow_pickle=False
)


print(
    "Loaded benchmark from:"
)

print(
    benchmark_path.resolve()
)

print()

print(
    "Available benchmark keys:"
)

print(
    benchmark_data.files
)

Loaded benchmark from:
C:\Users\liu.xuanc\Desktop\Code\Quantum-HOC\results_stage2\stage3_learner_benchmark.npz

Available benchmark keys:
['H_AB', 'H_BC', 'H_micro', 'K_strang_cubic', 'K_AB_renormalization', 'K_BC_renormalization', 'K_AC_phantom', 'J_AB', 'J_BC', 'epsilon_values', 'convention_version', 'reference_seed']


In [3]:
# ------------------------------------------------------------------
# Oracle layer
#
# These quantities may be used to:
#   1. generate synthetic experimental data;
#   2. evaluate the learner after fitting.
#
# They must not be used as regression targets or design-matrix inputs.
# ------------------------------------------------------------------

oracle_H_AB = benchmark_data[
    "H_AB"
]

oracle_H_BC = benchmark_data[
    "H_BC"
]

oracle_H_micro = benchmark_data[
    "H_micro"
]


oracle_K_strang_cubic = benchmark_data[
    "K_strang_cubic"
]

oracle_K_AB_renormalization = benchmark_data[
    "K_AB_renormalization"
]

oracle_K_BC_renormalization = benchmark_data[
    "K_BC_renormalization"
]

oracle_K_AC_phantom = benchmark_data[
    "K_AC_phantom"
]


oracle_J_AB = benchmark_data[
    "J_AB"
]

oracle_J_BC = benchmark_data[
    "J_BC"
]


benchmark_epsilon_values = benchmark_data[
    "epsilon_values"
]


benchmark_convention = str(
    benchmark_data[
        "convention_version"
    ]
)

In [4]:
# ------------------------------------------------------------------
# Learner-visible metadata
# ------------------------------------------------------------------

N = 3

A, B, C = 0, 1, 2


site_labels = {
    A: "A",
    B: "B",
    C: "C"
}


candidate_pairs = [
    "AB",
    "BC",
    "AC"
]


pair_sites = {
    "AB": (A, B),
    "BC": (B, C),
    "AC": (A, C)
}


# The learner is allowed to know:
#
#   - there are three qubits;
#   - candidate interactions are two-body Pauli bilinears;
#   - the protocol step epsilon;
#   - prepared input states;
#   - measured output observables.
#
# It is not told which candidate pairs are microscopic.

learner_visible_metadata = {

    "n_sites":
        N,

    "local_dimension":
        2,

    "candidate_pairs":
        tuple(candidate_pairs),

    "local_operator_axes":
        ("X", "Y", "Z"),

    "protocol_family":
        "Strang palindrome",

    "effective_time_normalization":
        "G_eff = i log(U) / epsilon"
}

In [5]:
# ------------------------------------------------------------------
# Benchmark integrity gate
# ------------------------------------------------------------------

matrix_dimension = 2**N


oracle_matrices = {

    "H_AB":
        oracle_H_AB,

    "H_BC":
        oracle_H_BC,

    "H_micro":
        oracle_H_micro,

    "K_strang_cubic":
        oracle_K_strang_cubic,

    "K_AB_renormalization":
        oracle_K_AB_renormalization,

    "K_BC_renormalization":
        oracle_K_BC_renormalization,

    "K_AC_phantom":
        oracle_K_AC_phantom
}


for name, matrix in oracle_matrices.items():

    assert matrix.shape == (
        matrix_dimension,
        matrix_dimension
    ), (
        f"{name} has incorrect shape: "
        f"{matrix.shape}"
    )


    hermiticity_error = np.linalg.norm(
        matrix
        -
        matrix.conj().T
    )


    assert hermiticity_error < 1e-10, (
        f"{name} is not Hermitian: "
        f"{hermiticity_error:.3e}"
    )


assert np.linalg.norm(
    oracle_H_micro
    -
    oracle_H_AB
    -
    oracle_H_BC
) < 1e-12


assert np.linalg.norm(
    oracle_K_strang_cubic
    -
    oracle_K_AB_renormalization
    -
    oracle_K_BC_renormalization
    -
    oracle_K_AC_phantom
) < 1e-12


assert np.linalg.norm(
    oracle_K_AC_phantom
) > 1e-6


assert (
    benchmark_convention
    ==
    cf.CONVENTION_VERSION
)


print(
    "Benchmark matrix integrity: PASS"
)

print(
    "Stage 2 / Stage 3 convention match: PASS"
)

print(
    "Oracle phantom target is nonzero: PASS"
)

print()

print(
    "Convention:"
)

print(
    benchmark_convention
)

print()

print(
    "Learner-visible metadata:"
)

display(
    pd.DataFrame(
        [
            {
                "field": key,
                "value": str(value)
            }

            for key, value
            in learner_visible_metadata.items()
        ]
    )
)

Benchmark matrix integrity: PASS
Stage 2 / Stage 3 convention match: PASS
Oracle phantom target is nonzero: PASS

Convention:
frozen-v1 (chronological: first-acting rightmost; L anti-Hermitian; K = iL)

Learner-visible metadata:


,field,value
0,n_sites,3
1,local_dimension,2
2,candidate_pairs,"('AB', 'BC', 'AC')"
3,local_operator_axes,"('X', 'Y', 'Z')"
4,protocol_family,Strang palindrome
5,effective_time_normalization,G_eff = i log(U) / epsilon


In [6]:
print(
    "INFORMATION BOUNDARY"
)

print(
    "  Learner may use:"
)

print(
    "    input states, output observables, epsilon, candidate basis"
)

print(
    "  Learner may not use:"
)

print(
    "    log(U), support projectors, microscopic edge labels,"
)

print(
    "    analytic BCH coefficients, oracle phantom target"
)

INFORMATION BOUNDARY
  Learner may use:
    input states, output observables, epsilon, candidate basis
  Learner may not use:
    log(U), support projectors, microscopic edge labels,
    analytic BCH coefficients, oracle phantom target


noiseless informationally complete observable dataset

In [7]:
# ------------------------------------------------------------------
# Informationally complete observable-data benchmark
# ------------------------------------------------------------------

data_seed = 20260809
data_rng = np.random.default_rng(
    data_seed
)

benchmark_epsilon = 0.08


# ==================================================================
# 1. Single-qubit Pauli eigenstates
# ==================================================================

single_qubit_states = {

    "+X":
        np.array(
            [1.0, 1.0],
            dtype=complex
        ) / np.sqrt(2),

    "-X":
        np.array(
            [1.0, -1.0],
            dtype=complex
        ) / np.sqrt(2),

    "+Y":
        np.array(
            [1.0, 1.0j],
            dtype=complex
        ) / np.sqrt(2),

    "-Y":
        np.array(
            [1.0, -1.0j],
            dtype=complex
        ) / np.sqrt(2),

    "+Z":
        np.array(
            [1.0, 0.0],
            dtype=complex
        ),

    "-Z":
        np.array(
            [0.0, 1.0],
            dtype=complex
        )
}


single_state_labels = list(
    single_qubit_states.keys()
)


# ==================================================================
# 2. All 6^3 product input states
# ==================================================================

input_state_rows = []


for local_labels in product(
    single_state_labels,
    repeat=N
):

    state_vector = cf.kron_all([

        single_qubit_states[
            label
        ]

        for label in local_labels
    ])


    input_state_rows.append({

        "label":
            "|".join(
                local_labels
            ),

        "local_labels":
            local_labels,

        "state":
            state_vector
    })


# Shuffle once, deterministically, before assigning splits.

state_permutation = data_rng.permutation(
    len(input_state_rows)
)


input_state_rows = [

    input_state_rows[index]

    for index in state_permutation
]


input_states = np.stack([

    row["state"]

    for row in input_state_rows
])


input_state_labels = np.array([

    row["label"]

    for row in input_state_rows
])


n_input_states = len(
    input_states
)


assert n_input_states == 6**N


# ==================================================================
# 3. Train / validation / test split
# ==================================================================

n_train = 108
n_validation = 54
n_test = 54


assert (
    n_train
    +
    n_validation
    +
    n_test
    ==
    n_input_states
)


split_labels = np.empty(
    n_input_states,
    dtype="<U10"
)


split_labels[
    :n_train
] = "train"


split_labels[
    n_train:
    n_train + n_validation
] = "validation"


split_labels[
    n_train + n_validation:
] = "test"


# ==================================================================
# 4. All nonidentity Pauli observables
# ==================================================================

pauli_operator_map = {
    "I": cf.I2,
    "X": cf.X,
    "Y": cf.Y,
    "Z": cf.Z
}


observable_labels = []
observable_matrices = []


for pauli_labels in product(
    ["I", "X", "Y", "Z"],
    repeat=N
):

    if all(
        label == "I"
        for label in pauli_labels
    ):
        continue


    observable_labels.append(
        "".join(
            pauli_labels
        )
    )


    observable_matrices.append(
        cf.kron_all([

            pauli_operator_map[
                label
            ]

            for label in pauli_labels
        ])
    )


observable_labels = np.array(
    observable_labels
)


observable_matrices = np.stack(
    observable_matrices
)


n_observables = len(
    observable_matrices
)


assert n_observables == 4**N - 1


# ==================================================================
# 5. Oracle protocol used only to generate observations
# ==================================================================

oracle_protocol_hams, oracle_protocol_areas = (
    cf.strang_lists(
        oracle_H_AB,
        oracle_H_BC,
        benchmark_epsilon
    )
)


oracle_U_benchmark = cf.protocol_unitary(
    oracle_protocol_hams,
    oracle_protocol_areas
)


# Row convention:
# input_states[state_index] is a ket stored as a row vector.

output_states = (
    oracle_U_benchmark
    @ input_states.T
).T


# ==================================================================
# 6. Observable expectation values
# ==================================================================

def batched_expectations(
    states,
    observables
):
    """
    states:
        shape (n_states, Hilbert_dimension)

    observables:
        shape (n_observables, Hilbert_dimension, Hilbert_dimension)

    returns:
        real expectation array of shape
        (n_states, n_observables)
    """

    complex_expectations = np.einsum(
        "si,oij,sj->so",
        states.conj(),
        observables,
        states,
        optimize=True
    )


    maximum_imaginary_part = np.max(
        np.abs(
            complex_expectations.imag
        )
    )


    assert (
        maximum_imaginary_part
        < 1e-12
    )


    return complex_expectations.real


input_expectations = batched_expectations(
    input_states,
    observable_matrices
)


output_expectations = batched_expectations(
    output_states,
    observable_matrices
)


observable_changes = (
    output_expectations
    -
    input_expectations
)


# ==================================================================
# 7. Informational-completeness diagnostics
# ==================================================================

input_projectors = np.stack([

    np.outer(
        state,
        state.conj()
    )

    for state in input_states
])


state_operator_rank = np.linalg.matrix_rank(

    input_projectors.reshape(
        n_input_states,
        -1
    ),

    tol=1e-10
)


measurement_operator_rank = np.linalg.matrix_rank(

    np.concatenate(
        [
            np.eye(
                2**N,
                dtype=complex
            )[None, :, :],

            observable_matrices
        ],
        axis=0
    ).reshape(
        4**N,
        -1
    ),

    tol=1e-10
)


state_norm_error = np.max(
    np.abs(
        np.sum(
            np.abs(
                input_states
            ) ** 2,
            axis=1
        )
        -
        1.0
    )
)


unitarity_error = np.linalg.norm(

    oracle_U_benchmark.conj().T
    @ oracle_U_benchmark

    -

    np.eye(
        2**N
    )
)


expectation_bound_violation = max(

    0.0,

    np.max(
        np.abs(
            output_expectations
        )
    )
    -
    1.0
)


observable_change_rms = np.sqrt(
    np.mean(
        observable_changes**2
    )
)


data_integrity_summary = pd.DataFrame({

    "quantity": [
        "number of input states",
        "number of observables",
        "train states",
        "validation states",
        "test states",
        "input-projector span rank",
        "measurement span rank",
        "maximum state-normalization error",
        "protocol unitarity error",
        "expectation-bound violation",
        "RMS observable change"
    ],

    "value": [
        n_input_states,
        n_observables,
        n_train,
        n_validation,
        n_test,
        state_operator_rank,
        measurement_operator_rank,
        state_norm_error,
        unitarity_error,
        expectation_bound_violation,
        observable_change_rms
    ]
})


display(
    data_integrity_summary
)


# ==================================================================
# 8. Split manifest
# ==================================================================

state_split_manifest = pd.DataFrame({

    "state_index":
        np.arange(
            n_input_states
        ),

    "state_label":
        input_state_labels,

    "split":
        split_labels
})


display(
    state_split_manifest.groupby(
        "split"
    ).size().rename(
        "number_of_states"
    ).reset_index()
)


# ==================================================================
# 9. Acceptance tests
# ==================================================================

assert (
    state_norm_error
    < 1e-12
)


assert (
    unitarity_error
    < 1e-12
)


assert (
    expectation_bound_violation
    < 1e-12
)


# Density matrices span the full 8 x 8 operator space.

assert (
    state_operator_rank
    == 4**N
)


# Identity plus 63 Pauli observables span the same space.

assert (
    measurement_operator_rank
    == 4**N
)


assert (
    observable_change_rms
    > 1e-3
)


assert (
    len(
        np.unique(
            input_state_labels
        )
    )
    ==
    n_input_states
)


print(
    "Product-state preparation set: PASS"
)

print(
    "Pauli measurement set: PASS"
)

print(
    "Input states are informationally complete: PASS"
)

print(
    "Measurements are informationally complete: PASS"
)

print(
    "Synthetic observable data integrity: PASS"
)

print()

print(
    "Benchmark epsilon:",
    benchmark_epsilon
)

print(
    "RMS observable change:",
    f"{observable_change_rms:.6f}"
)


# ==================================================================
# 10. Save learner-visible data only
# ==================================================================

learner_dataset_path = (
    stage3_result_dir
    / "learner_dataset_noiseless_eps_0p08.npz"
)


np.savez(

    learner_dataset_path,

    input_states=
        input_states,

    input_state_labels=
        input_state_labels,

    observable_matrices=
        observable_matrices,

    observable_labels=
        observable_labels,

    input_expectations=
        input_expectations,

    output_expectations=
        output_expectations,

    split_labels=
        split_labels,

    epsilon=
        benchmark_epsilon,

    data_seed=
        data_seed
)


state_split_manifest.to_csv(
    stage3_result_dir
    / "learner_state_split_manifest.csv",
    index=False
)


data_integrity_summary.to_csv(
    stage3_result_dir
    / "learner_data_integrity_summary.csv",
    index=False
)


print()

print(
    "Saved learner-visible dataset to:"
)

print(
    learner_dataset_path.resolve()
)

print()

print(
    "Saved keys:"
)

print([
    "input_states",
    "input_state_labels",
    "observable_matrices",
    "observable_labels",
    "input_expectations",
    "output_expectations",
    "split_labels",
    "epsilon",
    "data_seed"
])

,quantity,value
0,number of input states,2.160000e+02
1,number of observables,6.300000e+01
2,train states,1.080000e+02
3,validation states,5.400000e+01
4,test states,5.400000e+01
5,input-projector span rank,6.400000e+01
6,measurement span rank,6.400000e+01
7,maximum state-normalization error,4.440892e-16
8,protocol unitarity error,6.356034e-16
9,expectation-bound violation,0.000000e+00


,split,number_of_states
0,test,54
1,train,108
2,validation,54


Product-state preparation set: PASS
Pauli measurement set: PASS
Input states are informationally complete: PASS
Measurements are informationally complete: PASS
Synthetic observable data integrity: PASS

Benchmark epsilon: 0.08
RMS observable change: 0.093388

Saved learner-visible dataset to:
C:\Users\liu.xuanc\Desktop\Code\Quantum-HOC\results_stage3\learner_dataset_noiseless_eps_0p08.npz

Saved keys:
['input_states', 'input_state_labels', 'observable_matrices', 'observable_labels', 'input_expectations', 'output_expectations', 'split_labels', 'epsilon', 'data_seed']


M_0/M_1 candidate Hamiltonian

In [8]:
# ------------------------------------------------------------------
# Learner hypotheses and candidate Hamiltonian basis
# ------------------------------------------------------------------

learner_dataset = np.load(
    learner_dataset_path,
    allow_pickle=False
)


learner_input_states = learner_dataset[
    "input_states"
]

learner_observables = learner_dataset[
    "observable_matrices"
]

learner_output_expectations = learner_dataset[
    "output_expectations"
]

learner_split_labels = learner_dataset[
    "split_labels"
]

learner_epsilon = float(
    learner_dataset[
        "epsilon"
    ]
)


train_mask = (
    learner_split_labels
    == "train"
)

validation_mask = (
    learner_split_labels
    == "validation"
)

test_mask = (
    learner_split_labels
    == "test"
)


train_states = learner_input_states[
    train_mask
]

validation_states = learner_input_states[
    validation_mask
]

test_states = learner_input_states[
    test_mask
]


train_targets = learner_output_expectations[
    train_mask
]

validation_targets = learner_output_expectations[
    validation_mask
]

test_targets = learner_output_expectations[
    test_mask
]

In [9]:
# ------------------------------------------------------------------
# Complete two-body Pauli-bilinear basis
# ------------------------------------------------------------------

axis_labels = (
    "X",
    "Y",
    "Z"
)


axis_operators = {
    "X": cf.X,
    "Y": cf.Y,
    "Z": cf.Z
}


basis_rows = []
basis_matrices = []


for pair_name in candidate_pairs:

    site_i, site_j = pair_sites[
        pair_name
    ]


    for axis_i, axis_j in product(
        axis_labels,
        repeat=2
    ):

        basis_label = (
            f"{pair_name}:{axis_i}{axis_j}"
        )


        basis_matrix = cf.two_body(
            axis_operators[
                axis_i
            ],

            axis_operators[
                axis_j
            ],

            site_i,
            site_j,
            N
        )


        basis_rows.append({

            "basis_index":
                len(
                    basis_matrices
                ),

            "pair":
                pair_name,

            "axis_i":
                axis_i,

            "axis_j":
                axis_j,

            "basis_label":
                basis_label
        })


        basis_matrices.append(
            basis_matrix
        )


candidate_basis_metadata = pd.DataFrame(
    basis_rows
)


candidate_basis_matrices = np.stack(
    basis_matrices
)


n_candidate_parameters = len(
    candidate_basis_matrices
)


assert (
    n_candidate_parameters
    == 27
)


display(
    candidate_basis_metadata
)

,basis_index,pair,axis_i,axis_j,basis_label
0,0,AB,X,X,AB:XX
1,1,AB,X,Y,AB:XY
2,2,AB,X,Z,AB:XZ
3,3,AB,Y,X,AB:YX
4,4,AB,Y,Y,AB:YY
5,5,AB,Y,Z,AB:YZ
6,6,AB,Z,X,AB:ZX
7,7,AB,Z,Y,AB:ZY
8,8,AB,Z,Z,AB:ZZ
9,9,BC,X,X,BC:XX


In [10]:
# ------------------------------------------------------------------
# Model M0 and M1
# ------------------------------------------------------------------

M0_parameter_indices = (

    candidate_basis_metadata
    .index[
        candidate_basis_metadata[
            "pair"
        ].isin([
            "AB",
            "BC"
        ])
    ]
    .to_numpy()
)


M1_parameter_indices = np.arange(
    n_candidate_parameters
)


model_definitions = {

    "M0_micro_edges_only": {

        "parameter_indices":
            M0_parameter_indices,

        "basis_matrices":
            candidate_basis_matrices[
                M0_parameter_indices
            ],

        "basis_metadata":
            candidate_basis_metadata
            .iloc[
                M0_parameter_indices
            ]
            .reset_index(
                drop=True
            )
    },

    "M1_with_candidate_AC": {

        "parameter_indices":
            M1_parameter_indices,

        "basis_matrices":
            candidate_basis_matrices[
                M1_parameter_indices
            ],

        "basis_metadata":
            candidate_basis_metadata
            .iloc[
                M1_parameter_indices
            ]
            .reset_index(
                drop=True
            )
    }
}


model_summary = pd.DataFrame([

    {
        "model":
            model_name,

        "number_of_parameters":
            len(
                model[
                    "basis_matrices"
                ]
            ),

        "included_pairs":
            ", ".join(
                model[
                    "basis_metadata"
                ][
                    "pair"
                ]
                .drop_duplicates()
                .to_list()
            )
    }

    for model_name, model
    in model_definitions.items()
])


display(
    model_summary
)

,model,number_of_parameters,included_pairs
0,M0_micro_edges_only,18,"AB, BC"
1,M1_with_candidate_AC,27,"AB, BC, AC"


In [11]:
# ------------------------------------------------------------------
# Observable-data forward model
# ------------------------------------------------------------------

def generator_from_parameters(
    parameters,
    basis
):
    """
    Construct the Hermitian generator

        G(theta) = sum_k theta_k B_k.
    """

    parameters = np.asarray(
        parameters,
        dtype=float
    )


    if len(parameters) != len(basis):

        raise ValueError(
            "Parameter vector and basis must have equal length."
        )


    return np.einsum(
        "k,kij->ij",
        parameters,
        basis,
        optimize=True
    )


def predict_output_expectations(
    parameters,
    basis,
    input_states,
    epsilon
):
    """
    Learner prediction:

        U_model = exp(-i epsilon G(theta)),

    followed by the specified Pauli measurements.
    """

    generator = generator_from_parameters(
        parameters,
        basis
    )


    model_unitary = expm(
        -1j
        * epsilon
        * generator
    )


    model_output_states = (
        model_unitary
        @ input_states.T
    ).T


    return batched_expectations(
        model_output_states,
        learner_observables
    )


def observable_residual_vector(
    parameters,
    basis,
    input_states,
    targets,
    epsilon
):
    """
    Flattened observable residual used by least_squares.
    """

    predictions = predict_output_expectations(
        parameters,
        basis,
        input_states,
        epsilon
    )


    return (
        predictions
        -
        targets
    ).reshape(-1)

In [12]:
# ------------------------------------------------------------------
# Basis integrity
# ------------------------------------------------------------------

basis_hermiticity_errors = np.array([

    np.linalg.norm(
        basis_matrix
        -
        basis_matrix.conj().T
    )

    for basis_matrix
    in candidate_basis_matrices
])


basis_traces = np.array([

    np.trace(
        basis_matrix
    )

    for basis_matrix
    in candidate_basis_matrices
])


basis_gram_matrix = np.einsum(

    "aij,bij->ab",

    candidate_basis_matrices.conj(),
    candidate_basis_matrices,

    optimize=True
)


basis_gram_diagonal = np.diag(
    basis_gram_matrix
).real


basis_gram_off_diagonal = (

    basis_gram_matrix

    -

    np.diag(
        np.diag(
            basis_gram_matrix
        )
    )
)


basis_operator_rank = np.linalg.matrix_rank(

    candidate_basis_matrices.reshape(
        n_candidate_parameters,
        -1
    ),

    tol=1e-10
)

In [13]:
# ------------------------------------------------------------------
# Linear-response identifiability
# ------------------------------------------------------------------

n_train_states = len(
    train_states
)

n_measurements = len(
    learner_observables
)


full_sensitivity_tensor = np.empty(

    (
        n_train_states,
        n_measurements,
        n_candidate_parameters
    ),

    dtype=float
)


for parameter_index, basis_operator in enumerate(
    candidate_basis_matrices
):

    derivative_observables = (

        1j
        * learner_epsilon
        * (

            basis_operator[
                None,
                :,
                :
            ]

            @ learner_observables

            -

            learner_observables

            @ basis_operator[
                None,
                :,
                :
            ]
        )
    )


    full_sensitivity_tensor[
        :,
        :,
        parameter_index
    ] = batched_expectations(

        train_states,
        derivative_observables
    )


full_sensitivity_matrix = (
    full_sensitivity_tensor.reshape(
        -1,
        n_candidate_parameters
    )
)

In [14]:
identifiability_rows = []


for model_name, model in model_definitions.items():

    parameter_indices = model[
        "parameter_indices"
    ]


    model_sensitivity = (
        full_sensitivity_matrix[
            :,
            parameter_indices
        ]
    )


    singular_values = np.linalg.svd(
        model_sensitivity,
        compute_uv=False
    )


    sensitivity_rank = np.linalg.matrix_rank(
        model_sensitivity,
        tol=1e-10
    )


    condition_number = (

        singular_values[0]

        /

        singular_values[-1]
    )


    identifiability_rows.append({

        "model":
            model_name,

        "number_of_parameters":
            len(
                parameter_indices
            ),

        "sensitivity_rank":
            sensitivity_rank,

        "largest_singular_value":
            singular_values[0],

        "smallest_singular_value":
            singular_values[-1],

        "condition_number":
            condition_number
    })


model_identifiability = pd.DataFrame(
    identifiability_rows
)


display(
    model_identifiability
)

,model,number_of_parameters,sensitivity_rank,largest_singular_value,smallest_singular_value,condition_number
0,M0_micro_edges_only,18,18,3.351697,2.927202,1.145017
1,M1_with_candidate_AC,27,27,3.367916,2.898639,1.161895


In [15]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

assert (
    basis_hermiticity_errors.max()
    < 1e-12
)


assert (
    np.abs(
        basis_traces
    ).max()
    < 1e-12
)


assert (
    basis_operator_rank
    == 27
)


assert (
    np.abs(
        basis_gram_off_diagonal
    ).max()
    < 1e-12
)


assert (
    np.ptp(
        basis_gram_diagonal
    )
    < 1e-12
)


for _, row in model_identifiability.iterrows():

    assert (
        row[
            "sensitivity_rank"
        ]
        ==
        row[
            "number_of_parameters"
        ]
    )


    assert (
        row[
            "smallest_singular_value"
        ]
        > 1e-8
    )


print(
    "Candidate Hamiltonian basis integrity: PASS"
)

print(
    "M0 parameters are locally identifiable: PASS"
)

print(
    "M1 parameters are locally identifiable: PASS"
)

print()

print(
    "M0 parameter count:",
    len(
        M0_parameter_indices
    )
)

print(
    "M1 parameter count:",
    len(
        M1_parameter_indices
    )
)


candidate_basis_metadata.to_csv(
    stage3_result_dir
    / "learner_candidate_basis.csv",
    index=False
)


model_identifiability.to_csv(
    stage3_result_dir
    / "learner_model_identifiability.csv",
    index=False
)


np.savez(
    stage3_result_dir
    / "learner_candidate_basis.npz",

    basis_matrices=
        candidate_basis_matrices,

    basis_labels=
        candidate_basis_metadata[
            "basis_label"
        ].to_numpy(),

    M0_parameter_indices=
        M0_parameter_indices,

    M1_parameter_indices=
        M1_parameter_indices
)

Candidate Hamiltonian basis integrity: PASS
M0 parameters are locally identifiable: PASS
M1 parameters are locally identifiable: PASS

M0 parameter count: 18
M1 parameter count: 27


fitting M_0/M_1

In [16]:
# ------------------------------------------------------------------
# Fit M0 and M1 from observable data
# ------------------------------------------------------------------

from time import perf_counter


# ==================================================================
# 1. Learner-visible linear-response initialization
# ==================================================================

learner_input_expectations = learner_dataset[
    "input_expectations"
]


train_input_expectations = learner_input_expectations[
    train_mask
]


linear_response_target = (
    train_targets
    -
    train_input_expectations
).reshape(-1)


def linear_response_initialization(
    parameter_indices
):
    """
    Least-squares initialization from

        Delta <O> approximately J theta

    around the zero-generator point.
    """

    model_sensitivity = (
        full_sensitivity_matrix[
            :,
            parameter_indices
        ]
    )


    initial_parameters, *_ = np.linalg.lstsq(
        model_sensitivity,
        linear_response_target,
        rcond=None
    )


    return initial_parameters


initial_parameters = {

    model_name:
        linear_response_initialization(
            model[
                "parameter_indices"
            ]
        )

    for model_name, model
    in model_definitions.items()
}

In [17]:
# ==================================================================
# 2. Observable prediction metrics
# ==================================================================

def prediction_metrics(
    parameters,
    basis,
    states,
    targets,
    epsilon
):
    predictions = predict_output_expectations(
        parameters,
        basis,
        states,
        epsilon
    )


    residuals = (
        predictions
        -
        targets
    )


    return {

        "predictions":
            predictions,

        "rms":
            float(
                np.sqrt(
                    np.mean(
                        residuals**2
                    )
                )
            ),

        "mae":
            float(
                np.mean(
                    np.abs(
                        residuals
                    )
                )
            ),

        "maximum_absolute_error":
            float(
                np.max(
                    np.abs(
                        residuals
                    )
                )
            )
    }

In [18]:
# ==================================================================
# 3. Nonlinear observable-data fitting
# ==================================================================

learner_fits = {}


for model_name, model in model_definitions.items():

    basis = model[
        "basis_matrices"
    ]


    theta_initial = initial_parameters[
        model_name
    ]


    initial_train_metrics = prediction_metrics(
        theta_initial,
        basis,
        train_states,
        train_targets,
        learner_epsilon
    )


    start_time = perf_counter()


    fit_result = least_squares(

        observable_residual_vector,

        x0=
            theta_initial,

        args=(
            basis,
            train_states,
            train_targets,
            learner_epsilon
        ),

        method=
            "trf",

        jac=
            "2-point",

        x_scale=
            "jac",

        ftol=
            1e-12,

        xtol=
            1e-12,

        gtol=
            1e-12,

        max_nfev=
            600,

        verbose=
            0
    )


    elapsed_seconds = (
        perf_counter()
        -
        start_time
    )


    learner_fits[
        model_name
    ] = {

        "result":
            fit_result,

        "parameters":
            fit_result.x,

        "initial_train_rms":
            initial_train_metrics[
                "rms"
            ],

        "elapsed_seconds":
            elapsed_seconds
    }


    print(
        f"{model_name}:"
    )

    print(
        "  success:",
        fit_result.success
    )

    print(
        "  status:",
        fit_result.status
    )

    print(
        "  message:",
        fit_result.message
    )

    print(
        "  function evaluations:",
        fit_result.nfev
    )

    print(
        "  elapsed seconds:",
        f"{elapsed_seconds:.2f}"
    )

    print()

M0_micro_edges_only:
  success: True
  status: 2
  message: `ftol` termination condition is satisfied.
  function evaluations: 4
  elapsed seconds: 0.40

M1_with_candidate_AC:
  success: True
  status: 1
  message: `gtol` termination condition is satisfied.
  function evaluations: 4
  elapsed seconds: 0.75



In [19]:
# ==================================================================
# 4. Held-out evaluation
# ==================================================================

split_data = {

    "train": (
        train_states,
        train_targets
    ),

    "validation": (
        validation_states,
        validation_targets
    ),

    "test": (
        test_states,
        test_targets
    )
}


fit_metric_rows = []


for model_name, model in model_definitions.items():

    fitted_parameters = learner_fits[
        model_name
    ][
        "parameters"
    ]


    basis = model[
        "basis_matrices"
    ]


    for split_name, (
        split_states,
        split_targets
    ) in split_data.items():

        metrics = prediction_metrics(
            fitted_parameters,
            basis,
            split_states,
            split_targets,
            learner_epsilon
        )


        fit_metric_rows.append({

            "model":
                model_name,

            "split":
                split_name,

            "rms_error":
                metrics[
                    "rms"
                ],

            "mean_absolute_error":
                metrics[
                    "mae"
                ],

            "maximum_absolute_error":
                metrics[
                    "maximum_absolute_error"
                ]
        })


learner_fit_metrics = pd.DataFrame(
    fit_metric_rows
)


display(
    learner_fit_metrics
)

,model,split,rms_error,mean_absolute_error,maximum_absolute_error
0,M0_micro_edges_only,train,3.706171e-04,2.321954e-04,1.559901e-03
1,M0_micro_edges_only,validation,3.635396e-04,2.273515e-04,1.524098e-03
2,M0_micro_edges_only,test,3.668422e-04,2.297097e-04,1.553960e-03
3,M1_with_candidate_AC,train,9.642735e-17,5.960596e-17,7.216450e-16
4,M1_with_candidate_AC,validation,9.246563e-17,5.946946e-17,5.551115e-16
5,M1_with_candidate_AC,test,9.036256e-17,5.645828e-17,4.440892e-16


In [20]:
# ==================================================================
# 5. Optimization summary
# ==================================================================

optimization_summary = pd.DataFrame([

    {
        "model":
            model_name,

        "number_of_parameters":
            len(
                fit[
                    "parameters"
                ]
            ),

        "initial_train_rms":
            fit[
                "initial_train_rms"
            ],

        "final_train_rms":
            learner_fit_metrics.loc[

                (
                    learner_fit_metrics[
                        "model"
                    ]
                    ==
                    model_name
                )

                &

                (
                    learner_fit_metrics[
                        "split"
                    ]
                    ==
                    "train"
                ),

                "rms_error"

            ].iloc[0],

        "cost":
            fit[
                "result"
            ].cost,

        "optimality":
            fit[
                "result"
            ].optimality,

        "function_evaluations":
            fit[
                "result"
            ].nfev,

        "success":
            fit[
                "result"
            ].success,

        "elapsed_seconds":
            fit[
                "elapsed_seconds"
            ]
    }

    for model_name, fit
    in learner_fits.items()
])


display(
    optimization_summary
)

,model,number_of_parameters,initial_train_rms,final_train_rms,cost,optimality,function_evaluations,success,elapsed_seconds
0,M0_micro_edges_only,18,0.003022,3.706171e-04,4.672888e-04,8.967937e-10,4,True,0.404643
1,M1_with_candidate_AC,27,0.003026,9.642735e-17,3.163259e-29,1.797288e-15,4,True,0.748891


In [21]:
# ==================================================================
# 6. Learned effective-edge ledger
# ==================================================================

learned_coefficient_rows = []
learned_pair_rows = []


for model_name, model in model_definitions.items():

    fitted_parameters = learner_fits[
        model_name
    ][
        "parameters"
    ]


    metadata = model[
        "basis_metadata"
    ].copy()


    metadata[
        "coefficient"
    ] = fitted_parameters


    metadata[
        "model"
    ] = model_name


    learned_coefficient_rows.append(
        metadata
    )


    for pair_name in candidate_pairs:

        pair_mask_model = (
            metadata[
                "pair"
            ]
            ==
            pair_name
        )


        if pair_mask_model.any():

            pair_parameters = (
                fitted_parameters[
                    pair_mask_model.to_numpy()
                ]
            )


            pair_basis = (
                model[
                    "basis_matrices"
                ][
                    pair_mask_model.to_numpy()
                ]
            )


            learned_pair_operator = (
                generator_from_parameters(
                    pair_parameters,
                    pair_basis
                )
            )


            coefficient_l2_norm = np.linalg.norm(
                pair_parameters
            )


            operator_frobenius_norm = np.linalg.norm(
                learned_pair_operator
            )

        else:

            coefficient_l2_norm = 0.0
            operator_frobenius_norm = 0.0


        learned_pair_rows.append({

            "model":
                model_name,

            "pair":
                pair_name,

            "coefficient_l2_norm":
                coefficient_l2_norm,

            "operator_frobenius_norm":
                operator_frobenius_norm
        })


learned_coefficients = pd.concat(
    learned_coefficient_rows,
    ignore_index=True
)


learned_pair_ledger = pd.DataFrame(
    learned_pair_rows
)


display(
    learned_pair_ledger
)

,model,pair,coefficient_l2_norm,operator_frobenius_norm
0,M0_micro_edges_only,AB,1.756070,4.966916
1,M0_micro_edges_only,BC,1.744672,4.934677
2,M0_micro_edges_only,AC,0.000000,0.000000
3,M1_with_candidate_AC,AB,1.756084,4.966956
4,M1_with_candidate_AC,BC,1.744485,4.934149
5,M1_with_candidate_AC,AC,0.009713,0.027473


In [22]:
# ==================================================================
# 7. Model comparison
# ==================================================================

def split_rms(
    model_name,
    split_name
):
    return learner_fit_metrics.loc[

        (
            learner_fit_metrics[
                "model"
            ]
            ==
            model_name
        )

        &

        (
            learner_fit_metrics[
                "split"
            ]
            ==
            split_name
        ),

        "rms_error"

    ].iloc[0]


M0_test_rms = split_rms(
    "M0_micro_edges_only",
    "test"
)


M1_test_rms = split_rms(
    "M1_with_candidate_AC",
    "test"
)


heldout_improvement_factor = (
    M0_test_rms
    /
    max(
        M1_test_rms,
        1e-300
    )
)


M1_learned_AC_norm = learned_pair_ledger.loc[

    (
        learned_pair_ledger[
            "model"
        ]
        ==
        "M1_with_candidate_AC"
    )

    &

    (
        learned_pair_ledger[
            "pair"
        ]
        ==
        "AC"
    ),

    "operator_frobenius_norm"

].iloc[0]


model_comparison_summary = pd.DataFrame({

    "quantity": [
        "M0 test RMS",
        "M1 test RMS",
        "M0 / M1 held-out improvement factor",
        "M1 learned AC operator norm"
    ],

    "value": [
        M0_test_rms,
        M1_test_rms,
        heldout_improvement_factor,
        M1_learned_AC_norm
    ]
})


display(
    model_comparison_summary
)

,quantity,value
0,M0 test RMS,3.668422e-04
1,M1 test RMS,9.036256e-17
2,M0 / M1 held-out improvement factor,4.059670e+12
3,M1 learned AC operator norm,2.747327e-02


In [23]:
# ==================================================================
# 8. Acceptance tests
# ==================================================================

assert (
    optimization_summary[
        "success"
    ].all()
)


# M0 should leave a resolvable held-out residual because
# it forbids the effective AC interaction.

assert (
    M0_test_rms
    > 1e-6
)


# M1 should explain the noiseless observable data essentially exactly.

assert (
    M1_test_rms
    < 1e-6
)


# The candidate-AC model must improve out-of-sample prediction
# by at least two orders of magnitude.

assert (
    heldout_improvement_factor
    > 100
)


# The fitted M1 model must assign a nonzero interaction to AC.

assert (
    M1_learned_AC_norm
    > 1e-4
)


print(
    "M0 fit completed: PASS"
)

print(
    "M1 fit completed: PASS"
)

print(
    "M0 leaves a held-out systematic residual: PASS"
)

print(
    "M1 removes the held-out residual: PASS"
)

print(
    "Learner assigns a nonzero effective AC edge: PASS"
)

print()

print(
    "M0 test RMS:",
    f"{M0_test_rms:.6e}"
)

print(
    "M1 test RMS:",
    f"{M1_test_rms:.6e}"
)

print(
    "Held-out improvement factor:",
    f"{heldout_improvement_factor:.3e}"
)

print(
    "Learned AC operator norm:",
    f"{M1_learned_AC_norm:.6f}"
)

M0 fit completed: PASS
M1 fit completed: PASS
M0 leaves a held-out systematic residual: PASS
M1 removes the held-out residual: PASS
Learner assigns a nonzero effective AC edge: PASS

M0 test RMS: 3.668422e-04
M1 test RMS: 9.036256e-17
Held-out improvement factor: 4.060e+12
Learned AC operator norm: 0.027473


Oracle-side post-fit validation

In [24]:
# ------------------------------------------------------------------
# Oracle-side post-fit validation
#
# IMPORTANT:
# Oracle quantities are used only after the observable-data fit.
# They are not used to initialize, train, or select M0/M1.
# ------------------------------------------------------------------

M1_model_name = "M1_with_candidate_AC"

M1_fitted_parameters = learner_fits[
    M1_model_name
][
    "parameters"
]

M1_basis = model_definitions[
    M1_model_name
][
    "basis_matrices"
]

M1_metadata = model_definitions[
    M1_model_name
][
    "basis_metadata"
]


# ==================================================================
# 1. Reconstruct the learned full effective generator
# ==================================================================

learned_G_eff = generator_from_parameters(
    M1_fitted_parameters,
    M1_basis
)


# ==================================================================
# 2. Construct the exact finite-step oracle generator
#
# This is permitted only for post-fit evaluation.
# ==================================================================

oracle_L_eff = cf.log_unitary(
    oracle_U_benchmark
)

oracle_G_eff = (
    1j
    * oracle_L_eff
    / learner_epsilon
)


oracle_G_hermiticity_error = np.linalg.norm(
    oracle_G_eff
    -
    oracle_G_eff.conj().T
)

oracle_G_trace = np.trace(
    oracle_G_eff
)


assert (
    oracle_G_hermiticity_error
    < 1e-12
)


# The traceless candidate basis fixes the otherwise unobservable
# global-phase / identity gauge.

assert (
    abs(
        oracle_G_trace
    )
    < 1e-10
)


# ==================================================================
# 3. Utility functions
# ==================================================================

def relative_frobenius_error(
    estimate,
    target
):
    return (

        np.linalg.norm(
            estimate
            -
            target
        )

        /

        max(
            np.linalg.norm(
                target
            ),
            1e-300
        )
    )


def hilbert_schmidt_alignment(
    estimate,
    target
):
    """
    Returns:
        cosine:
            normalized Hilbert-Schmidt alignment;

        amplitude:
            best scalar alpha in estimate ~ alpha * target;

        direction_residual:
            relative residual after removing the best-fit amplitude.
    """

    estimate_norm = np.linalg.norm(
        estimate
    )

    target_norm = np.linalg.norm(
        target
    )


    overlap = np.vdot(
        target,
        estimate
    ).real


    cosine = (

        overlap

        /

        max(
            estimate_norm
            * target_norm,
            1e-300
        )
    )


    amplitude = (

        overlap

        /

        max(
            target_norm**2,
            1e-300
        )
    )


    aligned_estimate = (
        amplitude
        *
        target
    )


    direction_residual = (

        np.linalg.norm(
            estimate
            -
            aligned_estimate
        )

        /

        max(
            estimate_norm,
            1e-300
        )
    )


    return (
        cosine,
        amplitude,
        direction_residual
    )


# ==================================================================
# 4. Project learned and oracle generators onto candidate pairs
# ==================================================================

learned_pair_operators = {}
oracle_pair_operators = {}


for pair_name, sites in pair_sites.items():

    pair_mask = (
        M1_metadata[
            "pair"
        ]
        ==
        pair_name
    ).to_numpy()


    learned_pair_operators[
        pair_name
    ] = generator_from_parameters(

        M1_fitted_parameters[
            pair_mask
        ],

        M1_basis[
            pair_mask
        ]
    )


    oracle_pair_operators[
        pair_name
    ] = cf.exact_support_component(

        oracle_G_eff,

        set(
            sites
        ),

        local_dim=2,
        n_sites=N
    )


oracle_pair_sum = sum(

    oracle_pair_operators.values(),

    np.zeros_like(
        oracle_G_eff
    )
)


oracle_pair_closure_error = relative_frobenius_error(
    oracle_pair_sum,
    oracle_G_eff
)


# ==================================================================
# 5. Full-generator comparison
# ==================================================================

full_generator_relative_error = relative_frobenius_error(
    learned_G_eff,
    oracle_G_eff
)


(
    full_generator_cosine,
    full_generator_amplitude,
    full_generator_direction_residual
) = hilbert_schmidt_alignment(
    learned_G_eff,
    oracle_G_eff
)


# ==================================================================
# 6. Pair-by-pair operator validation
# ==================================================================

pair_validation_rows = []


for pair_name in candidate_pairs:

    learned_operator = learned_pair_operators[
        pair_name
    ]

    oracle_operator = oracle_pair_operators[
        pair_name
    ]


    (
        cosine,
        amplitude,
        direction_residual
    ) = hilbert_schmidt_alignment(
        learned_operator,
        oracle_operator
    )


    pair_validation_rows.append({

        "pair":
            pair_name,

        "learned_operator_norm":
            np.linalg.norm(
                learned_operator
            ),

        "oracle_operator_norm":
            np.linalg.norm(
                oracle_operator
            ),

        "relative_operator_error":
            relative_frobenius_error(
                learned_operator,
                oracle_operator
            ),

        "Hilbert_Schmidt_cosine":
            cosine,

        "fitted_amplitude":
            amplitude,

        "direction_residual":
            direction_residual
    })


pair_operator_validation = pd.DataFrame(
    pair_validation_rows
)


display(
    pair_operator_validation
)


# ==================================================================
# 7. Extract 3 x 3 Pauli coupling matrices
# ==================================================================

axis_to_index = {
    axis: index
    for index, axis
    in enumerate(
        axis_labels
    )
}


def coupling_matrix_from_parameter_vector(
    parameters,
    metadata,
    pair_name
):
    """
    Convert the nine coefficients on one pair into a 3 x 3 matrix

        J[mu, nu] sigma_mu sigma_nu.
    """

    coupling_matrix = np.zeros(
        (3, 3),
        dtype=float
    )


    pair_rows = metadata.loc[
        metadata[
            "pair"
        ]
        ==
        pair_name
    ]


    for row_index, row in pair_rows.iterrows():

        mu = axis_to_index[
            row[
                "axis_i"
            ]
        ]

        nu = axis_to_index[
            row[
                "axis_j"
            ]
        ]


        coupling_matrix[
            mu,
            nu
        ] = parameters[
            row_index
        ]


    return coupling_matrix


def project_operator_onto_candidate_basis(
    operator,
    basis
):
    """
    Hilbert-Schmidt projection onto the orthogonal Pauli basis.
    """

    basis_norm_squared = np.einsum(
        "kij,kij->k",
        basis.conj(),
        basis,
        optimize=True
    ).real


    overlaps = np.einsum(
        "kij,ij->k",
        basis.conj(),
        operator,
        optimize=True
    ).real


    return (
        overlaps
        /
        basis_norm_squared
    )


oracle_exact_parameters = project_operator_onto_candidate_basis(
    oracle_G_eff,
    candidate_basis_matrices
)


leading_order_G_eff = (

    oracle_H_micro

    +

    learner_epsilon**2
    * oracle_K_strang_cubic
)


leading_order_parameters = project_operator_onto_candidate_basis(
    leading_order_G_eff,
    candidate_basis_matrices
)


learned_coupling_matrices = {}
oracle_coupling_matrices = {}
leading_coupling_matrices = {}


for pair_name in candidate_pairs:

    learned_coupling_matrices[
        pair_name
    ] = coupling_matrix_from_parameter_vector(

        M1_fitted_parameters,
        M1_metadata,
        pair_name
    )


    oracle_coupling_matrices[
        pair_name
    ] = coupling_matrix_from_parameter_vector(

        oracle_exact_parameters,
        candidate_basis_metadata,
        pair_name
    )


    leading_coupling_matrices[
        pair_name
    ] = coupling_matrix_from_parameter_vector(

        leading_order_parameters,
        candidate_basis_metadata,
        pair_name
    )


# ==================================================================
# 8. AC coupling-matrix comparison
# ==================================================================

learned_J_AC = learned_coupling_matrices[
    "AC"
]

oracle_J_AC_exact = oracle_coupling_matrices[
    "AC"
]

oracle_J_AC_leading = leading_coupling_matrices[
    "AC"
]


AC_coupling_rows = []


for mu_index, mu in enumerate(
    axis_labels
):

    for nu_index, nu in enumerate(
        axis_labels
    ):

        AC_coupling_rows.append({

            "channel":
                f"{mu}{nu}",

            "learned":
                learned_J_AC[
                    mu_index,
                    nu_index
                ],

            "exact_finite_step":
                oracle_J_AC_exact[
                    mu_index,
                    nu_index
                ],

            "leading_epsilon2_prediction":
                oracle_J_AC_leading[
                    mu_index,
                    nu_index
                ],

            "learned_minus_exact":
                (
                    learned_J_AC[
                        mu_index,
                        nu_index
                    ]

                    -

                    oracle_J_AC_exact[
                        mu_index,
                        nu_index
                    ]
                ),

            "exact_minus_leading":
                (
                    oracle_J_AC_exact[
                        mu_index,
                        nu_index
                    ]

                    -

                    oracle_J_AC_leading[
                        mu_index,
                        nu_index
                    ]
                )
        })


AC_coupling_validation = pd.DataFrame(
    AC_coupling_rows
)


display(
    AC_coupling_validation
)

,pair,learned_operator_norm,oracle_operator_norm,relative_operator_error,Hilbert_Schmidt_cosine,fitted_amplitude,direction_residual
0,AB,4.966956,4.966956,7.061309e-16,1.0,1.0,6.672042e-16
1,BC,4.934149,4.934149,7.496516e-16,1.0,1.0,7.327458e-16
2,AC,0.027473,0.027473,9.607081e-14,1.0,1.0,8.333917e-14


,channel,learned,exact_finite_step,leading_epsilon2_prediction,learned_minus_exact,exact_minus_leading
0,XX,-0.001963,-0.001963,-0.001947,2.120699e-16,-0.000016
1,XY,0.002998,0.002998,0.002985,2.142383e-16,0.000013
2,XZ,-0.003382,-0.003382,-0.003359,4.692427e-16,-0.000023
3,YX,0.003171,0.003171,0.003149,-3.981190e-16,0.000022
4,YY,-0.003365,-0.003365,-0.003349,-3.139849e-16,-0.000016
5,YZ,0.000687,0.000687,0.000685,-1.052760e-16,0.000002
6,ZX,-0.006053,-0.006053,-0.006020,2.905662e-16,-0.000033
7,ZY,-0.000342,-0.000342,-0.000347,-2.748995e-16,0.000005
8,ZZ,-0.003384,-0.003384,-0.003362,3.495468e-16,-0.000022


In [25]:
# ==================================================================
# 9. Leading-order versus exact finite-step AC
# ==================================================================

oracle_AC_exact = oracle_pair_operators[
    "AC"
]

oracle_AC_leading = (

    learner_epsilon**2
    *
    oracle_K_AC_phantom
)


learned_AC = learned_pair_operators[
    "AC"
]


AC_learned_exact_error = relative_frobenius_error(
    learned_AC,
    oracle_AC_exact
)


AC_exact_leading_error = relative_frobenius_error(
    oracle_AC_exact,
    oracle_AC_leading
)


(
    AC_learned_exact_cosine,
    AC_learned_exact_amplitude,
    AC_learned_exact_direction_residual
) = hilbert_schmidt_alignment(
    learned_AC,
    oracle_AC_exact
)


(
    AC_exact_leading_cosine,
    AC_exact_leading_amplitude,
    AC_exact_leading_direction_residual
) = hilbert_schmidt_alignment(
    oracle_AC_exact,
    oracle_AC_leading
)


oracle_validation_summary = pd.DataFrame({

    "quantity": [
        "full learned/exact relative error",
        "full learned/exact cosine",
        "full learned/exact amplitude",
        "oracle pair-closure error",
        "learned AC norm",
        "exact finite-step AC norm",
        "leading epsilon^2 AC norm",
        "learned/exact AC relative error",
        "learned/exact AC cosine",
        "learned/exact AC amplitude",
        "learned/exact AC direction residual",
        "exact/leading AC relative error",
        "exact/leading AC cosine",
        "exact/leading AC amplitude",
        "exact/leading AC direction residual"
    ],

    "value": [
        full_generator_relative_error,
        full_generator_cosine,
        full_generator_amplitude,
        oracle_pair_closure_error,

        np.linalg.norm(
            learned_AC
        ),

        np.linalg.norm(
            oracle_AC_exact
        ),

        np.linalg.norm(
            oracle_AC_leading
        ),

        AC_learned_exact_error,
        AC_learned_exact_cosine,
        AC_learned_exact_amplitude,
        AC_learned_exact_direction_residual,

        AC_exact_leading_error,
        AC_exact_leading_cosine,
        AC_exact_leading_amplitude,
        AC_exact_leading_direction_residual
    ]
})


display(
    oracle_validation_summary
)

,quantity,value
0,full learned/exact relative error,1.516300e-15
1,full learned/exact cosine,1.000000e+00
2,full learned/exact amplitude,1.000000e+00
3,oracle pair-closure error,1.281974e-15
4,learned AC norm,2.747327e-02
5,exact finite-step AC norm,2.747327e-02
6,leading epsilon^2 AC norm,2.731399e-02
7,learned/exact AC relative error,9.607081e-14
8,learned/exact AC cosine,1.000000e+00
9,learned/exact AC amplitude,1.000000e+00


In [26]:
# ==================================================================
# 10. Acceptance tests
# ==================================================================

# The exact finite-step generator is entirely pairwise.

assert (
    oracle_pair_closure_error
    < 1e-10
)


# Observable-data learner recovers the complete effective generator.

assert (
    full_generator_relative_error
    < 1e-10
)


assert (
    full_generator_cosine
    > 1 - 1e-12
)


assert (
    abs(
        full_generator_amplitude
        -
        1.0
    )
    < 1e-10
)


# Every pair sector is recovered, including the phantom AC edge.

assert (
    pair_operator_validation[
        "relative_operator_error"
    ].max()
    < 1e-10
)


assert (
    pair_operator_validation[
        "Hilbert_Schmidt_cosine"
    ].min()
    > 1 - 1e-12
)


# AC matrix learned from observable data equals the exact
# finite-step effective AC interaction.

assert (
    AC_learned_exact_error
    < 1e-10
)


assert (
    AC_learned_exact_cosine
    > 1 - 1e-12
)


assert (
    AC_learned_exact_direction_residual
    < 1e-10
)


assert (
    np.max(
        np.abs(
            learned_J_AC
            -
            oracle_J_AC_exact
        )
    )
    < 1e-10
)


# The cubic BCH prediction should already approximate the exact
# finite-step AC edge at epsilon = 0.08, with higher-order deviations.

assert (
    AC_exact_leading_error
    < 2e-2
)


assert (
    AC_exact_leading_cosine
    > 0.999
)


print(
    "Exact finite-step generator is pairwise: PASS"
)

print(
    "Observable-data learner recovers full generator: PASS"
)

print(
    "All effective pair sectors recovered: PASS"
)

print(
    "Phantom AC operator direction recovered: PASS"
)

print(
    "Phantom AC coupling matrix recovered: PASS"
)

print(
    "Finite-step AC agrees with epsilon^2 BCH prediction: PASS"
)

print()

print(
    "Full generator relative error:",
    f"{full_generator_relative_error:.3e}"
)

print(
    "Learned/exact AC relative error:",
    f"{AC_learned_exact_error:.3e}"
)

print(
    "Learned/exact AC cosine:",
    f"{AC_learned_exact_cosine:.12f}"
)

print(
    "Exact/leading AC relative error:",
    f"{AC_exact_leading_error:.6e}"
)

print()

print(
    "Learned AC coupling matrix:"
)

print(
    learned_J_AC
)

print()

print(
    "Exact finite-step AC coupling matrix:"
)

print(
    oracle_J_AC_exact
)

Exact finite-step generator is pairwise: PASS
Observable-data learner recovers full generator: PASS
All effective pair sectors recovered: PASS
Phantom AC operator direction recovered: PASS
Phantom AC coupling matrix recovered: PASS
Finite-step AC agrees with epsilon^2 BCH prediction: PASS

Full generator relative error: 1.516e-15
Learned/exact AC relative error: 9.607e-14
Learned/exact AC cosine: 1.000000000000
Exact/leading AC relative error: 5.961310e-03

Learned AC coupling matrix:
[[-0.001963  0.002998 -0.003382]
 [ 0.003171 -0.003365  0.000687]
 [-0.006053 -0.000342 -0.003384]]

Exact finite-step AC coupling matrix:
[[-0.001963  0.002998 -0.003382]
 [ 0.003171 -0.003365  0.000687]
 [-0.006053 -0.000342 -0.003384]]


In [27]:
# ==================================================================
# 11. Save post-fit validation
# ==================================================================

pair_operator_validation.to_csv(
    stage3_result_dir
    / "learner_pair_operator_validation.csv",
    index=False
)


AC_coupling_validation.to_csv(
    stage3_result_dir
    / "learner_AC_coupling_validation.csv",
    index=False
)


oracle_validation_summary.to_csv(
    stage3_result_dir
    / "learner_oracle_validation_summary.csv",
    index=False
)


np.savez(
    stage3_result_dir
    / "learner_oracle_validation.npz",

    learned_G_eff=
        learned_G_eff,

    oracle_G_eff=
        oracle_G_eff,

    learned_AC=
        learned_AC,

    oracle_AC_exact=
        oracle_AC_exact,

    oracle_AC_leading=
        oracle_AC_leading,

    learned_J_AC=
        learned_J_AC,

    oracle_J_AC_exact=
        oracle_J_AC_exact,

    oracle_J_AC_leading=
        oracle_J_AC_leading,

    epsilon=
        learner_epsilon
)

Learner-side scaling law

In [28]:
# ------------------------------------------------------------------
# Learner-side epsilon scaling
#
# Expected:
#
#   ||G_AC_hat||              ~ epsilon^2
#   held-out residual of M0   ~ epsilon^3
#
# M1 is fitted independently at every epsilon from observable data.
# ------------------------------------------------------------------

scaling_epsilon_values = np.array([
    0.02,
    0.03,
    0.04,
    0.06,
    0.08,
    0.10,
    0.12,
    0.16
])


scaling_rows = []

M0_scaling_parameters = []
M1_scaling_parameters = []


warm_start_parameters = {

    "M0_micro_edges_only":
        None,

    "M1_with_candidate_AC":
        None
}


# Input-state expectation values do not depend on epsilon.

scaling_train_input_expectations = (
    learner_input_expectations[
        train_mask
    ]
)


for epsilon in scaling_epsilon_values:

    # ==============================================================
    # 1. Oracle data generation only
    # ==============================================================

    protocol_hams, protocol_areas = cf.strang_lists(
        oracle_H_AB,
        oracle_H_BC,
        epsilon
    )


    oracle_U_epsilon = cf.protocol_unitary(
        protocol_hams,
        protocol_areas
    )


    branch_distance, _ = cf.unitary_branch_distance(
        oracle_U_epsilon
    )


    output_states_epsilon = (
        oracle_U_epsilon
        @ learner_input_states.T
    ).T


    output_expectations_epsilon = batched_expectations(
        output_states_epsilon,
        learner_observables
    )


    train_targets_epsilon = (
        output_expectations_epsilon[
            train_mask
        ]
    )

    validation_targets_epsilon = (
        output_expectations_epsilon[
            validation_mask
        ]
    )

    test_targets_epsilon = (
        output_expectations_epsilon[
            test_mask
        ]
    )


    # ==============================================================
    # 2. Epsilon-specific linear-response initialization
    #
    # The zero-generator sensitivity is linear in epsilon.
    # ==============================================================

    sensitivity_epsilon = (

        full_sensitivity_matrix

        *

        epsilon
        / learner_epsilon
    )


    linear_target_epsilon = (

        train_targets_epsilon

        -

        scaling_train_input_expectations

    ).reshape(-1)


    fitted_models_epsilon = {}


    for model_name, model in model_definitions.items():

        parameter_indices = model[
            "parameter_indices"
        ]

        basis = model[
            "basis_matrices"
        ]


        model_sensitivity = (
            sensitivity_epsilon[
                :,
                parameter_indices
            ]
        )


        theta_linear, *_ = np.linalg.lstsq(
            model_sensitivity,
            linear_target_epsilon,
            rcond=None
        )


        # ----------------------------------------------------------
        # Choose between:
        #
        #   1. fresh linear-response initialization;
        #   2. previous epsilon's observable-data fit.
        #
        # No oracle parameters are used.
        # ----------------------------------------------------------

        initialization_candidates = [
            theta_linear
        ]


        if (
            warm_start_parameters[
                model_name
            ]
            is not None
        ):

            initialization_candidates.append(

                warm_start_parameters[
                    model_name
                ]
            )


        initialization_rms_values = []


        for initialization in initialization_candidates:

            initialization_metrics = prediction_metrics(
                initialization,
                basis,
                train_states,
                train_targets_epsilon,
                epsilon
            )


            initialization_rms_values.append(
                initialization_metrics[
                    "rms"
                ]
            )


        best_initialization_index = int(
            np.argmin(
                initialization_rms_values
            )
        )


        theta_initial = (
            initialization_candidates[
                best_initialization_index
            ]
        )


        fit_result = least_squares(

            observable_residual_vector,

            x0=
                theta_initial,

            args=(
                basis,
                train_states,
                train_targets_epsilon,
                epsilon
            ),

            method=
                "trf",

            jac=
                "2-point",

            x_scale=
                "jac",

            ftol=
                1e-12,

            xtol=
                1e-12,

            gtol=
                1e-12,

            max_nfev=
                500,

            verbose=
                0
        )


        warm_start_parameters[
            model_name
        ] = fit_result.x.copy()


        train_metrics = prediction_metrics(
            fit_result.x,
            basis,
            train_states,
            train_targets_epsilon,
            epsilon
        )


        validation_metrics = prediction_metrics(
            fit_result.x,
            basis,
            validation_states,
            validation_targets_epsilon,
            epsilon
        )


        test_metrics = prediction_metrics(
            fit_result.x,
            basis,
            test_states,
            test_targets_epsilon,
            epsilon
        )


        fitted_models_epsilon[
            model_name
        ] = {

            "result":
                fit_result,

            "parameters":
                fit_result.x.copy(),

            "train_rms":
                train_metrics["rms"],

            "validation_rms":
                validation_metrics["rms"],

            "test_rms":
                test_metrics["rms"]
        }


    # ==============================================================
    # 3. Extract learner-inferred AC operator
    # ==============================================================

    M1_parameters_epsilon = (
        fitted_models_epsilon[
            "M1_with_candidate_AC"
        ][
            "parameters"
        ]
    )


    M1_AC_mask = (

        model_definitions[
            "M1_with_candidate_AC"
        ][
            "basis_metadata"
        ][
            "pair"
        ]

        ==
        "AC"

    ).to_numpy()


    learned_AC_epsilon = generator_from_parameters(

        M1_parameters_epsilon[
            M1_AC_mask
        ],

        model_definitions[
            "M1_with_candidate_AC"
        ][
            "basis_matrices"
        ][
            M1_AC_mask
        ]
    )


    learned_AC_norm_epsilon = np.linalg.norm(
        learned_AC_epsilon
    )


    # ==============================================================
    # 4. Oracle-side post-fit evaluation only
    # ==============================================================

    oracle_L_epsilon = cf.log_unitary(
        oracle_U_epsilon
    )


    oracle_G_epsilon = (
        1j
        * oracle_L_epsilon
        / epsilon
    )


    oracle_AC_epsilon = cf.exact_support_component(
        oracle_G_epsilon,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    learned_exact_AC_error = relative_frobenius_error(
        learned_AC_epsilon,
        oracle_AC_epsilon
    )


    (
        learned_exact_AC_cosine,
        _,
        learned_exact_AC_direction_residual
    ) = hilbert_schmidt_alignment(
        learned_AC_epsilon,
        oracle_AC_epsilon
    )


    M0_fit = fitted_models_epsilon[
        "M0_micro_edges_only"
    ]


    M1_fit = fitted_models_epsilon[
        "M1_with_candidate_AC"
    ]


    scaling_rows.append({

        "epsilon":
            epsilon,

        "branch_distance":
            branch_distance,

        "M0_success":
            M0_fit[
                "result"
            ].success,

        "M1_success":
            M1_fit[
                "result"
            ].success,

        "M0_nfev":
            M0_fit[
                "result"
            ].nfev,

        "M1_nfev":
            M1_fit[
                "result"
            ].nfev,

        "M0_train_rms":
            M0_fit[
                "train_rms"
            ],

        "M0_validation_rms":
            M0_fit[
                "validation_rms"
            ],

        "M0_test_rms":
            M0_fit[
                "test_rms"
            ],

        "M1_train_rms":
            M1_fit[
                "train_rms"
            ],

        "M1_validation_rms":
            M1_fit[
                "validation_rms"
            ],

        "M1_test_rms":
            M1_fit[
                "test_rms"
            ],

        "learned_AC_norm":
            learned_AC_norm_epsilon,

        "learned_AC_norm_over_epsilon2":
            (
                learned_AC_norm_epsilon
                /
                epsilon**2
            ),

        "oracle_AC_norm":
            np.linalg.norm(
                oracle_AC_epsilon
            ),

        "oracle_AC_norm_over_epsilon2":
            (
                np.linalg.norm(
                    oracle_AC_epsilon
                )
                /
                epsilon**2
            ),

        "learned_exact_AC_relative_error":
            learned_exact_AC_error,

        "learned_exact_AC_cosine":
            learned_exact_AC_cosine,

        "learned_exact_AC_direction_residual":
            learned_exact_AC_direction_residual,

        "M0_test_rms_over_epsilon3":
            (
                M0_fit[
                    "test_rms"
                ]
                /
                epsilon**3
            )
    })


    M0_scaling_parameters.append(

        M0_fit[
            "parameters"
        ]
    )


    M1_scaling_parameters.append(

        M1_fit[
            "parameters"
        ]
    )


learner_scaling_results = pd.DataFrame(
    scaling_rows
)


display(
    learner_scaling_results
)

,epsilon,branch_distance,M0_success,M1_success,M0_nfev,M1_nfev,M0_train_rms,M0_validation_rms,M0_test_rms,M1_train_rms,M1_validation_rms,M1_test_rms,learned_AC_norm,learned_AC_norm_over_epsilon2,oracle_AC_norm,oracle_AC_norm_over_epsilon2,learned_exact_AC_relative_error,learned_exact_AC_cosine,learned_exact_AC_direction_residual,M0_test_rms_over_epsilon3
0,0.02,3.072253,True,True,4,3,0.000006,0.000006,0.000006,8.257109e-17,8.366764e-17,8.692356e-17,0.001708,4.269365,0.001708,4.269365,8.501719e-12,1.0,8.501748e-12,0.715794
1,0.03,3.037597,True,True,3,3,0.000020,0.000019,0.000019,1.042409e-16,1.073339e-16,1.005598e-16,0.003844,4.271307,0.003844,4.271307,3.140900e-12,1.0,3.006214e-12,0.715861
2,0.04,3.002957,True,True,4,3,0.000046,0.000045,0.000046,8.698358e-17,9.367849e-17,9.250342e-17,0.006838,4.274027,0.006838,4.274027,7.673256e-13,1.0,6.758079e-13,0.715951
3,0.06,2.933748,True,True,7,3,0.000156,0.000153,0.000155,2.849850e-16,2.962340e-16,2.885753e-16,0.015414,4.281803,0.015414,4.281803,9.976454e-13,1.0,9.548536e-13,0.716188
4,0.08,2.864671,True,True,4,3,0.000371,0.000364,0.000367,8.069098e-16,8.242584e-16,7.985026e-16,0.027473,4.292698,0.027473,4.292698,1.192932e-12,1.0,1.192922e-12,0.716489
5,0.10,2.795772,True,True,4,3,0.000724,0.000711,0.000717,4.321430e-15,4.246134e-15,4.352830e-15,0.043067,4.306724,0.043067,4.306724,2.662215e-12,1.0,2.481812e-12,0.716830
6,0.12,2.727095,True,True,6,3,0.001251,0.001232,0.001239,5.745762e-15,5.706960e-15,5.658021e-15,0.062264,4.323890,0.062264,4.323890,1.485963e-12,1.0,1.238523e-12,0.717186
7,0.16,2.590604,True,True,4,4,0.002968,0.002933,0.002940,1.119699e-16,1.107331e-16,1.149527e-16,0.111813,4.367693,0.111813,4.367693,1.756591e-14,1.0,1.712899e-14,0.717816


In [29]:
# ------------------------------------------------------------------
# Scaling fits
# ------------------------------------------------------------------

asymptotic_scaling_mask = (

    learner_scaling_results[
        "epsilon"
    ]

    <= 0.08
)


scaling_epsilon_fit = (

    learner_scaling_results.loc[
        asymptotic_scaling_mask,
        "epsilon"
    ].to_numpy()
)


learned_AC_scaling_exponent = np.polyfit(

    np.log(
        scaling_epsilon_fit
    ),

    np.log(

        learner_scaling_results.loc[
            asymptotic_scaling_mask,
            "learned_AC_norm"
        ].to_numpy()
    ),

    deg=1
)[0]


oracle_AC_scaling_exponent = np.polyfit(

    np.log(
        scaling_epsilon_fit
    ),

    np.log(

        learner_scaling_results.loc[
            asymptotic_scaling_mask,
            "oracle_AC_norm"
        ].to_numpy()
    ),

    deg=1
)[0]


M0_residual_scaling_exponent = np.polyfit(

    np.log(
        scaling_epsilon_fit
    ),

    np.log(

        learner_scaling_results.loc[
            asymptotic_scaling_mask,
            "M0_test_rms"
        ].to_numpy()
    ),

    deg=1
)[0]


smallest_epsilon_row = (

    learner_scaling_results
    .sort_values(
        "epsilon"
    )
    .iloc[0]
)


analytic_K_AC_norm = np.linalg.norm(
    oracle_K_AC_phantom
)


learner_scaling_summary = pd.DataFrame({

    "quantity": [
        "learned AC norm exponent",
        "oracle AC norm exponent",
        "M0 held-out residual exponent",
        "analytic cubic K_AC norm",
        "smallest-epsilon learned AC / epsilon^2",
        "smallest-epsilon oracle AC / epsilon^2",
        "maximum M1 test RMS",
        "maximum learned/exact AC relative error",
        "minimum learned/exact AC cosine",
        "minimum branch distance"
    ],

    "value": [
        learned_AC_scaling_exponent,
        oracle_AC_scaling_exponent,
        M0_residual_scaling_exponent,
        analytic_K_AC_norm,

        smallest_epsilon_row[
            "learned_AC_norm_over_epsilon2"
        ],

        smallest_epsilon_row[
            "oracle_AC_norm_over_epsilon2"
        ],

        learner_scaling_results[
            "M1_test_rms"
        ].max(),

        learner_scaling_results[
            "learned_exact_AC_relative_error"
        ].max(),

        learner_scaling_results[
            "learned_exact_AC_cosine"
        ].min(),

        learner_scaling_results[
            "branch_distance"
        ].min()
    ]
})


display(
    learner_scaling_summary
)

,quantity,value
0,learned AC norm exponent,2.003811e+00
1,oracle AC norm exponent,2.003811e+00
2,M0 held-out residual exponent,3.000685e+00
3,analytic cubic K_AC norm,4.267811e+00
4,smallest-epsilon learned AC / epsilon^2,4.269365e+00
5,smallest-epsilon oracle AC / epsilon^2,4.269365e+00
6,maximum M1 test RMS,5.658021e-15
7,maximum learned/exact AC relative error,8.501719e-12
8,minimum learned/exact AC cosine,1.000000e+00
9,minimum branch distance,2.590604e+00


In [30]:
# ------------------------------------------------------------------
# Acceptance tests
# ------------------------------------------------------------------

assert (
    learner_scaling_results[
        "M0_success"
    ].all()
)


assert (
    learner_scaling_results[
        "M1_success"
    ].all()
)


# M1 must recover each noiseless finite-step dataset.

assert (
    learner_scaling_results[
        "M1_test_rms"
    ].max()
    < 1e-10
)


# Post-fit oracle check: AC is recovered at every epsilon.

assert (
    learner_scaling_results[
        "learned_exact_AC_relative_error"
    ].max()
    < 1e-8
)


assert (
    learner_scaling_results[
        "learned_exact_AC_cosine"
    ].min()
    > 1 - 1e-10
)


assert (
    learner_scaling_results[
        "learned_exact_AC_direction_residual"
    ].max()
    < 1e-8
)


# Effective AC interaction scales as epsilon^2.

assert (
    1.95
    <
    learned_AC_scaling_exponent
    <
    2.05
)


assert (
    1.95
    <
    oracle_AC_scaling_exponent
    <
    2.05
)


# Missing AC produces an observable residual of order epsilon^3.

assert (
    2.85
    <
    M0_residual_scaling_exponent
    <
    3.15
)


# Small-epsilon scaled AC norm converges to the cubic BCH target.

assert (

    abs(

        smallest_epsilon_row[
            "learned_AC_norm_over_epsilon2"
        ]

        -

        analytic_K_AC_norm
    )

    /

    analytic_K_AC_norm

    < 1e-3
)


assert (
    learner_scaling_results[
        "branch_distance"
    ].min()
    > 1.0
)


print(
    "All epsilon-dependent M0 fits completed: PASS"
)

print(
    "All epsilon-dependent M1 fits completed: PASS"
)

print(
    "M1 recovers every finite-step generator: PASS"
)

print(
    "Learner-inferred AC scales as epsilon^2: PASS"
)

print(
    "M0 held-out residual scales as epsilon^3: PASS"
)

print(
    "Scaled learned AC converges to cubic BCH target: PASS"
)

print()

print(
    "Learned AC exponent:",
    f"{learned_AC_scaling_exponent:.6f}"
)

print(
    "Oracle AC exponent:",
    f"{oracle_AC_scaling_exponent:.6f}"
)

print(
    "M0 residual exponent:",
    f"{M0_residual_scaling_exponent:.6f}"
)

print(
    "Analytic ||K_AC||:",
    f"{analytic_K_AC_norm:.6f}"
)

print(
    "Smallest-epsilon learned ||G_AC|| / epsilon^2:",
    f"{smallest_epsilon_row['learned_AC_norm_over_epsilon2']:.6f}"
)

All epsilon-dependent M0 fits completed: PASS
All epsilon-dependent M1 fits completed: PASS
M1 recovers every finite-step generator: PASS
Learner-inferred AC scales as epsilon^2: PASS
M0 held-out residual scales as epsilon^3: PASS
Scaled learned AC converges to cubic BCH target: PASS

Learned AC exponent: 2.003811
Oracle AC exponent: 2.003811
M0 residual exponent: 3.000685
Analytic ||K_AC||: 4.267811
Smallest-epsilon learned ||G_AC|| / epsilon^2: 4.269365


In [31]:
# ------------------------------------------------------------------
# Save learner scaling results
# ------------------------------------------------------------------

learner_scaling_results.to_csv(
    stage3_result_dir
    / "learner_epsilon_scaling.csv",
    index=False
)


learner_scaling_summary.to_csv(
    stage3_result_dir
    / "learner_epsilon_scaling_summary.csv",
    index=False
)


np.savez(
    stage3_result_dir
    / "learner_epsilon_scaling_parameters.npz",

    epsilon_values=
        scaling_epsilon_values,

    M0_parameters=
        np.stack(
            M0_scaling_parameters
        ),

    M1_parameters=
        np.stack(
            M1_scaling_parameters
        ),

    learned_AC_scaling_exponent=
        learned_AC_scaling_exponent,

    oracle_AC_scaling_exponent=
        oracle_AC_scaling_exponent,

    M0_residual_scaling_exponent=
        M0_residual_scaling_exponent
)

lower observable budget

In [32]:
# ------------------------------------------------------------------
# Observable-budget compression
#
# Design information:
#   - candidate Hamiltonian basis
#   - allowed input states and observables
#   - protocol epsilon
#
# Not used for design:
#   - output expectation values
#   - oracle generator
#   - microscopic topology
#   - analytic phantom target
# ------------------------------------------------------------------

from scipy.linalg import qr


full_train_target_flat = (
    train_targets.reshape(-1)
)

full_train_input_flat = (
    train_input_expectations.reshape(-1)
)


n_full_train_measurements = (
    full_sensitivity_matrix.shape[0]
)


assert (
    n_full_train_measurements
    ==
    n_train
    * n_observables
)


# ==================================================================
# 1. Learner-visible measurement design
# ==================================================================

# First obtain 27 linearly independent rows using pivoted QR.

_, _, qr_pivots = qr(
    full_sensitivity_matrix.T,
    mode="economic",
    pivoting=True
)


independent_seed_rows = np.asarray(
    qr_pivots[:n_candidate_parameters],
    dtype=int
)


seed_sensitivity = (
    full_sensitivity_matrix[
        independent_seed_rows
    ]
)


assert (
    np.linalg.matrix_rank(
        seed_sensitivity,
        tol=1e-10
    )
    ==
    n_candidate_parameters
)


# Rank-27 left singular space gives parameter-sensitivity leverage
# scores for all state-observable pairs.

sensitivity_left_vectors, _, _ = np.linalg.svd(
    full_sensitivity_matrix,
    full_matrices=False
)


row_leverage_scores = np.sum(
    sensitivity_left_vectors[
        :,
        :n_candidate_parameters
    ] ** 2,
    axis=1
)


leverage_order = np.argsort(
    -row_leverage_scores
)


seed_row_set = set(
    independent_seed_rows.tolist()
)


remaining_rows = np.array(
    [
        row_index

        for row_index in leverage_order

        if row_index not in seed_row_set
    ],
    dtype=int
)


measurement_selection_order = np.concatenate(
    [
        independent_seed_rows,
        remaining_rows
    ]
)


assert (
    len(
        np.unique(
            measurement_selection_order
        )
    )
    ==
    n_full_train_measurements
)


# Nested measurement budgets.

measurement_budgets = np.array([
    27,
    54,
    108,
    216,
    432,
    864,
    1728,
    3402,
    6804
])


assert (
    measurement_budgets[-1]
    ==
    n_full_train_measurements
)

In [33]:
# ==================================================================
# 2. Selected-data forward model
# ==================================================================

def selected_observable_residual_vector(
    parameters,
    basis,
    input_states,
    full_targets_flat,
    epsilon,
    selected_flat_indices
):
    """
    Compute the full model predictions internally, but return residuals
    only for the selected state-observable measurement settings.
    """

    predictions_flat = (
        predict_output_expectations(
            parameters,
            basis,
            input_states,
            epsilon
        )
        .reshape(-1)
    )


    return (

        predictions_flat[
            selected_flat_indices
        ]

        -

        full_targets_flat[
            selected_flat_indices
        ]
    )


def selected_train_rms(
    parameters,
    basis,
    selected_flat_indices
):
    residual = selected_observable_residual_vector(

        parameters,
        basis,
        train_states,
        full_train_target_flat,
        learner_epsilon,
        selected_flat_indices
    )


    return float(
        np.sqrt(
            np.mean(
                residual**2
            )
        )
    )

In [34]:
# ==================================================================
# 3. Fit nested measurement budgets
# ==================================================================

budget_rows = []

budget_parameter_records = {
    "M0_micro_edges_only": [],
    "M1_with_candidate_AC": []
}


previous_budget_parameters = {
    "M0_micro_edges_only": None,
    "M1_with_candidate_AC": None
}


for measurement_budget in measurement_budgets:

    selected_indices = (
        measurement_selection_order[
            :measurement_budget
        ]
    )


    selected_linear_target = (

        full_train_target_flat[
            selected_indices
        ]

        -

        full_train_input_flat[
            selected_indices
        ]
    )


    fitted_budget_models = {}


    for model_name, model in model_definitions.items():

        parameter_indices = model[
            "parameter_indices"
        ]

        basis = model[
            "basis_matrices"
        ]


        selected_sensitivity = (

            full_sensitivity_matrix[
                selected_indices
            ][
                :,
                parameter_indices
            ]
        )


        sensitivity_singular_values = np.linalg.svd(
            selected_sensitivity,
            compute_uv=False
        )


        sensitivity_rank = np.linalg.matrix_rank(
            selected_sensitivity,
            tol=1e-10
        )


        sensitivity_condition = (

            sensitivity_singular_values[0]

            /

            max(
                sensitivity_singular_values[-1],
                1e-300
            )
        )


        theta_linear, *_ = np.linalg.lstsq(
            selected_sensitivity,
            selected_linear_target,
            rcond=None
        )


        initialization_candidates = [
            theta_linear
        ]


        if (
            previous_budget_parameters[
                model_name
            ]
            is not None
        ):

            initialization_candidates.append(

                previous_budget_parameters[
                    model_name
                ]
            )


        initialization_errors = [

            selected_train_rms(
                candidate,
                basis,
                selected_indices
            )

            for candidate in initialization_candidates
        ]


        theta_initial = initialization_candidates[
            int(
                np.argmin(
                    initialization_errors
                )
            )
        ]


        fit_result = least_squares(

            selected_observable_residual_vector,

            x0=
                theta_initial,

            args=(
                basis,
                train_states,
                full_train_target_flat,
                learner_epsilon,
                selected_indices
            ),

            method=
                "trf",

            jac=
                "2-point",

            x_scale=
                "jac",

            ftol=
                1e-12,

            xtol=
                1e-12,

            gtol=
                1e-12,

            max_nfev=
                500,

            verbose=
                0
        )


        previous_budget_parameters[
            model_name
        ] = fit_result.x.copy()


        full_test_metrics = prediction_metrics(
            fit_result.x,
            basis,
            test_states,
            test_targets,
            learner_epsilon
        )


        fitted_budget_models[
            model_name
        ] = {

            "parameters":
                fit_result.x.copy(),

            "success":
                fit_result.success,

            "nfev":
                fit_result.nfev,

            "sensitivity_rank":
                sensitivity_rank,

            "sensitivity_condition":
                sensitivity_condition,

            "selected_train_rms":
                selected_train_rms(
                    fit_result.x,
                    basis,
                    selected_indices
                ),

            "full_test_rms":
                full_test_metrics[
                    "rms"
                ]
        }


        budget_parameter_records[
            model_name
        ].append(
            fit_result.x.copy()
        )


    # ==============================================================
    # 4. Extract learned AC and perform post-fit oracle evaluation
    # ==============================================================

    M0_budget_fit = fitted_budget_models[
        "M0_micro_edges_only"
    ]

    M1_budget_fit = fitted_budget_models[
        "M1_with_candidate_AC"
    ]


    M1_budget_parameters = (
        M1_budget_fit[
            "parameters"
        ]
    )


    M1_budget_AC = generator_from_parameters(

        M1_budget_parameters[
            M1_AC_mask
        ],

        model_definitions[
            "M1_with_candidate_AC"
        ][
            "basis_matrices"
        ][
            M1_AC_mask
        ]
    )


    M1_budget_AC_norm = np.linalg.norm(
        M1_budget_AC
    )


    M1_budget_AC_error = relative_frobenius_error(
        M1_budget_AC,
        oracle_AC_exact
    )


    (
        M1_budget_AC_cosine,
        _,
        M1_budget_AC_direction_residual
    ) = hilbert_schmidt_alignment(
        M1_budget_AC,
        oracle_AC_exact
    )


    heldout_improvement = (

        M0_budget_fit[
            "full_test_rms"
        ]

        /

        max(
            M1_budget_fit[
                "full_test_rms"
            ],
            1e-300
        )
    )


    budget_rows.append({

        "measurement_budget":
            measurement_budget,

        "budget_fraction":
            (
                measurement_budget
                /
                n_full_train_measurements
            ),

        "M0_sensitivity_rank":
            M0_budget_fit[
                "sensitivity_rank"
            ],

        "M1_sensitivity_rank":
            M1_budget_fit[
                "sensitivity_rank"
            ],

        "M0_condition_number":
            M0_budget_fit[
                "sensitivity_condition"
            ],

        "M1_condition_number":
            M1_budget_fit[
                "sensitivity_condition"
            ],

        "M0_success":
            M0_budget_fit[
                "success"
            ],

        "M1_success":
            M1_budget_fit[
                "success"
            ],

        "M0_nfev":
            M0_budget_fit[
                "nfev"
            ],

        "M1_nfev":
            M1_budget_fit[
                "nfev"
            ],

        "M0_selected_train_rms":
            M0_budget_fit[
                "selected_train_rms"
            ],

        "M1_selected_train_rms":
            M1_budget_fit[
                "selected_train_rms"
            ],

        "M0_full_test_rms":
            M0_budget_fit[
                "full_test_rms"
            ],

        "M1_full_test_rms":
            M1_budget_fit[
                "full_test_rms"
            ],

        "heldout_improvement_factor":
            heldout_improvement,

        "learned_AC_norm":
            M1_budget_AC_norm,

        "learned_AC_relative_error":
            M1_budget_AC_error,

        "learned_AC_cosine":
            M1_budget_AC_cosine,

        "learned_AC_direction_residual":
            M1_budget_AC_direction_residual
    })


observable_budget_results = pd.DataFrame(
    budget_rows
)


display(
    observable_budget_results
)

,measurement_budget,budget_fraction,M0_sensitivity_rank,M1_sensitivity_rank,M0_condition_number,M1_condition_number,M0_success,M1_success,M0_nfev,M1_nfev,M0_selected_train_rms,M1_selected_train_rms,M0_full_test_rms,M1_full_test_rms,heldout_improvement_factor,learned_AC_norm,learned_AC_relative_error,learned_AC_cosine,learned_AC_direction_residual
0,27,0.003968,18,27,2.414214,2.414214,True,True,5,4,0.000383,8.328516e-15,0.000382,8.122440e-15,4.703013e+10,0.027473,7.886889e-12,1.0,7.430435e-12
1,54,0.007937,18,27,8.054195,8.054195,True,True,5,1,0.000293,8.114727e-15,0.000388,8.122440e-15,4.773661e+10,0.027473,7.886889e-12,1.0,7.430435e-12
2,108,0.015873,18,27,6.129918,6.971008,True,True,5,1,0.000245,8.411721e-15,0.000397,8.122440e-15,4.888495e+10,0.027473,7.886889e-12,1.0,7.430435e-12
3,216,0.031746,18,27,6.455802,6.536674,True,True,4,1,0.000211,7.967502e-15,0.000387,8.122440e-15,4.767779e+10,0.027473,7.886889e-12,1.0,7.430435e-12
4,432,0.063492,18,27,8.904022,9.065028,True,True,4,1,0.000266,7.577658e-15,0.000371,8.122440e-15,4.567945e+10,0.027473,7.886889e-12,1.0,7.430435e-12
5,864,0.126984,18,27,4.961593,11.129719,True,True,4,1,0.000337,8.485489e-15,0.000372,8.122440e-15,4.578099e+10,0.027473,7.886889e-12,1.0,7.430435e-12
6,1728,0.253968,18,27,3.073871,3.712264,True,True,5,1,0.000347,9.041816e-15,0.000371,8.122440e-15,4.567639e+10,0.027473,7.886889e-12,1.0,7.430435e-12
7,3402,0.500000,18,27,2.182197,2.215957,True,True,5,1,0.000401,9.102331e-15,0.000367,8.122440e-15,4.518010e+10,0.027473,7.886889e-12,1.0,7.430435e-12
8,6804,1.000000,18,27,1.145017,1.161895,True,True,4,2,0.000371,9.530813e-17,0.000367,9.210485e-17,3.982875e+12,0.027473,1.130689e-13,1.0,9.589576e-14


In [35]:
# ==================================================================
# 5. Compact-design criterion
# ==================================================================

observable_budget_results[
    "successful_phantom_recovery"
] = (

    (
        observable_budget_results[
            "M1_sensitivity_rank"
        ]
        ==
        27
    )

    &

    (
        observable_budget_results[
            "M1_full_test_rms"
        ]
        <
        1e-6
    )

    &

    (
        observable_budget_results[
            "learned_AC_relative_error"
        ]
        <
        5e-2
    )

    &

    (
        observable_budget_results[
            "learned_AC_cosine"
        ]
        >
        0.999
    )

    &

    (
        observable_budget_results[
            "heldout_improvement_factor"
        ]
        >
        10
    )
)


successful_budget_rows = observable_budget_results.loc[
    observable_budget_results[
        "successful_phantom_recovery"
    ]
]


assert (
    len(
        successful_budget_rows
    )
    >
    0
)


minimum_successful_budget = int(

    successful_budget_rows[
        "measurement_budget"
    ].min()
)


minimum_successful_row = (

    successful_budget_rows
    .sort_values(
        "measurement_budget"
    )
    .iloc[0]
)


observable_budget_summary = pd.DataFrame({

    "quantity": [
        "full training measurement count",
        "minimum successful measurement budget",
        "minimum successful budget fraction",
        "minimum-budget M1 test RMS",
        "minimum-budget M0 test RMS",
        "minimum-budget improvement factor",
        "minimum-budget AC relative error",
        "minimum-budget AC cosine",
        "full-budget M1 test RMS",
        "full-budget AC relative error"
    ],

    "value": [
        n_full_train_measurements,

        minimum_successful_budget,

        minimum_successful_row[
            "budget_fraction"
        ],

        minimum_successful_row[
            "M1_full_test_rms"
        ],

        minimum_successful_row[
            "M0_full_test_rms"
        ],

        minimum_successful_row[
            "heldout_improvement_factor"
        ],

        minimum_successful_row[
            "learned_AC_relative_error"
        ],

        minimum_successful_row[
            "learned_AC_cosine"
        ],

        observable_budget_results.iloc[-1][
            "M1_full_test_rms"
        ],

        observable_budget_results.iloc[-1][
            "learned_AC_relative_error"
        ]
    ]
})


display(
    observable_budget_summary
)

,quantity,value
0,full training measurement count,6.804000e+03
1,minimum successful measurement budget,2.700000e+01
2,minimum successful budget fraction,3.968254e-03
3,minimum-budget M1 test RMS,8.122440e-15
4,minimum-budget M0 test RMS,3.819994e-04
5,minimum-budget improvement factor,4.703013e+10
6,minimum-budget AC relative error,7.886889e-12
7,minimum-budget AC cosine,1.000000e+00
8,full-budget M1 test RMS,9.210485e-17
9,full-budget AC relative error,1.130689e-13


In [36]:
# ==================================================================
# 6. Acceptance tests
# ==================================================================

assert (
    observable_budget_results[
        "M0_success"
    ].all()
)


assert (
    observable_budget_results[
        "M1_success"
    ].all()
)


assert (
    observable_budget_results.iloc[0][
        "M1_sensitivity_rank"
    ]
    ==
    27
)


# Full-budget result must reproduce the previous learner fit.

assert (
    observable_budget_results.iloc[-1][
        "M1_full_test_rms"
    ]
    <
    1e-10
)


assert (
    observable_budget_results.iloc[-1][
        "learned_AC_relative_error"
    ]
    <
    1e-8
)


assert (
    observable_budget_results.iloc[-1][
        "learned_AC_cosine"
    ]
    >
    1 - 1e-10
)


print(
    "Learner-visible measurement design: PASS"
)

print(
    "All compressed M0 fits completed: PASS"
)

print(
    "All compressed M1 fits completed: PASS"
)

print(
    "Full-budget fit reproduces exact learner result: PASS"
)

print(
    "A compressed design recovers the phantom edge: PASS"
)

print()

print(
    "Full measurement budget:",
    n_full_train_measurements
)

print(
    "Minimum successful budget:",
    minimum_successful_budget
)

print(
    "Retained fraction:",
    f"{minimum_successful_row['budget_fraction']:.6f}"
)

print(
    "Minimum-budget M1 test RMS:",
    f"{minimum_successful_row['M1_full_test_rms']:.6e}"
)

print(
    "Minimum-budget AC relative error:",
    f"{minimum_successful_row['learned_AC_relative_error']:.6e}"
)


observable_budget_results.to_csv(
    stage3_result_dir
    / "learner_observable_budget_ablation.csv",
    index=False
)


observable_budget_summary.to_csv(
    stage3_result_dir
    / "learner_observable_budget_summary.csv",
    index=False
)


compact_selected_indices = (

    measurement_selection_order[
        :minimum_successful_budget
    ]
)


compact_state_indices = (
    compact_selected_indices
    //
    n_observables
)


compact_observable_indices = (
    compact_selected_indices
    %
    n_observables
)


compact_measurement_manifest = pd.DataFrame({

    "selection_rank":
        np.arange(
            minimum_successful_budget
        ),

    "flat_train_index":
        compact_selected_indices,

    "train_state_index":
        compact_state_indices,

    "state_label":
        input_state_labels[
            train_mask
        ][
            compact_state_indices
        ],

    "observable_index":
        compact_observable_indices,

    "observable_label":
        observable_labels[
            compact_observable_indices
        ],

    "leverage_score":
        row_leverage_scores[
            compact_selected_indices
        ]
})


compact_measurement_manifest.to_csv(
    stage3_result_dir
    / "learner_compact_measurement_manifest.csv",
    index=False
)


np.savez(
    stage3_result_dir
    / "learner_observable_budget_parameters.npz",

    measurement_budgets=
        measurement_budgets,

    measurement_selection_order=
        measurement_selection_order,

    minimum_successful_budget=
        minimum_successful_budget,

    compact_selected_indices=
        compact_selected_indices,

    M0_parameters=
        np.stack(
            budget_parameter_records[
                "M0_micro_edges_only"
            ]
        ),

    M1_parameters=
        np.stack(
            budget_parameter_records[
                "M1_with_candidate_AC"
            ]
        )
)

Learner-visible measurement design: PASS
All compressed M0 fits completed: PASS
All compressed M1 fits completed: PASS
Full-budget fit reproduces exact learner result: PASS
A compressed design recovers the phantom edge: PASS

Full measurement budget: 6804
Minimum successful budget: 27
Retained fraction: 0.003968
Minimum-budget M1 test RMS: 8.122440e-15
Minimum-budget AC relative error: 7.886889e-12


Finite-shot robustness map

In [37]:
# ------------------------------------------------------------------
# Finite-shot robustness of compressed phantom learning
# ------------------------------------------------------------------

shot_noise_seed = 20260810
shot_noise_rng = np.random.default_rng(
    shot_noise_seed
)


noise_measurement_budgets = np.array([
    27,
    54,
    108,
    216
])


shot_counts = np.array([
    10_000,
    100_000,
    1_000_000
])


n_noise_replicates = 20


# ==================================================================
# 1. Pauli shot-noise sampler
# ==================================================================

def sample_pauli_expectations(
    exact_expectations,
    n_shots,
    rng
):
    """
    Each Pauli observable has outcomes +/-1.

    If its exact expectation is y, then

        P(+1) = (1 + y) / 2.
    """

    exact_expectations = np.asarray(
        exact_expectations,
        dtype=float
    )


    plus_probabilities = np.clip(
        (
            1.0
            +
            exact_expectations
        )
        /
        2.0,
        0.0,
        1.0
    )


    plus_counts = rng.binomial(
        n=n_shots,
        p=plus_probabilities
    )


    return (
        2.0
        *
        plus_counts
        /
        n_shots

        -

        1.0
    )


# ==================================================================
# 2. Efficient forward model for selected state-observable pairs
# ==================================================================

def predict_selected_expectations(
    parameters,
    basis,
    input_states,
    epsilon,
    selected_state_indices,
    selected_observable_indices
):
    generator = generator_from_parameters(
        parameters,
        basis
    )


    unitary = expm(
        -1j
        *
        epsilon
        *
        generator
    )


    output_states = (
        unitary
        @ input_states.T
    ).T


    selected_states = output_states[
        selected_state_indices
    ]


    selected_observables = learner_observables[
        selected_observable_indices
    ]


    expectations = np.einsum(
        "bi,bij,bj->b",
        selected_states.conj(),
        selected_observables,
        selected_states,
        optimize=True
    )


    assert (
        np.max(
            np.abs(
                expectations.imag
            )
        )
        <
        1e-10
    )


    return expectations.real


def noisy_selected_residual(
    parameters,
    basis,
    input_states,
    epsilon,
    selected_state_indices,
    selected_observable_indices,
    noisy_targets
):
    predictions = predict_selected_expectations(
        parameters,
        basis,
        input_states,
        epsilon,
        selected_state_indices,
        selected_observable_indices
    )


    return (
        predictions
        -
        noisy_targets
    )


# ==================================================================
# 3. Bootstrap loop
# ==================================================================

shot_noise_rows = []


for measurement_budget in noise_measurement_budgets:

    selected_flat_indices = (
        measurement_selection_order[
            :measurement_budget
        ]
    )


    selected_state_indices = (
        selected_flat_indices
        //
        n_observables
    )


    selected_observable_indices = (
        selected_flat_indices
        %
        n_observables
    )


    exact_selected_targets = (
        full_train_target_flat[
            selected_flat_indices
        ]
    )


    exact_selected_inputs = (
        full_train_input_flat[
            selected_flat_indices
        ]
    )


    for n_shots in shot_counts:

        for replicate in range(
            n_noise_replicates
        ):

            noisy_selected_targets = (
                sample_pauli_expectations(
                    exact_selected_targets,
                    n_shots,
                    shot_noise_rng
                )
            )


            noisy_linear_target = (
                noisy_selected_targets
                -
                exact_selected_inputs
            )


            fitted_models = {}


            for model_name, model in model_definitions.items():

                parameter_indices = model[
                    "parameter_indices"
                ]


                basis = model[
                    "basis_matrices"
                ]


                selected_sensitivity = (

                    full_sensitivity_matrix[
                        selected_flat_indices
                    ][
                        :,
                        parameter_indices
                    ]
                )


                theta_initial, *_ = np.linalg.lstsq(
                    selected_sensitivity,
                    noisy_linear_target,
                    rcond=None
                )


                fit_result = least_squares(

                    noisy_selected_residual,

                    x0=
                        theta_initial,

                    args=(
                        basis,
                        train_states,
                        learner_epsilon,
                        selected_state_indices,
                        selected_observable_indices,
                        noisy_selected_targets
                    ),

                    method=
                        "trf",

                    jac=
                        "2-point",

                    x_scale=
                        "jac",

                    ftol=
                        1e-11,

                    xtol=
                        1e-11,

                    gtol=
                        1e-11,

                    max_nfev=
                        400,

                    verbose=
                        0
                )


                test_metrics = prediction_metrics(
                    fit_result.x,
                    basis,
                    test_states,
                    test_targets,
                    learner_epsilon
                )


                fitted_models[
                    model_name
                ] = {

                    "parameters":
                        fit_result.x,

                    "success":
                        fit_result.success,

                    "nfev":
                        fit_result.nfev,

                    "test_rms":
                        test_metrics[
                            "rms"
                        ]
                }


            # ------------------------------------------------------
            # Post-fit AC evaluation
            # ------------------------------------------------------

            M0_noise_fit = fitted_models[
                "M0_micro_edges_only"
            ]


            M1_noise_fit = fitted_models[
                "M1_with_candidate_AC"
            ]


            M1_noise_parameters = (
                M1_noise_fit[
                    "parameters"
                ]
            )


            learned_AC_noise = generator_from_parameters(

                M1_noise_parameters[
                    M1_AC_mask
                ],

                model_definitions[
                    "M1_with_candidate_AC"
                ][
                    "basis_matrices"
                ][
                    M1_AC_mask
                ]
            )


            AC_relative_error = relative_frobenius_error(
                learned_AC_noise,
                oracle_AC_exact
            )


            (
                AC_cosine,
                AC_amplitude,
                AC_direction_residual
            ) = hilbert_schmidt_alignment(
                learned_AC_noise,
                oracle_AC_exact
            )


            heldout_improvement = (

                M0_noise_fit[
                    "test_rms"
                ]

                /

                max(
                    M1_noise_fit[
                        "test_rms"
                    ],
                    1e-300
                )
            )


            phantom_recovered = (

                M1_noise_fit[
                    "test_rms"
                ]
                <
                M0_noise_fit[
                    "test_rms"
                ]

                and

                AC_relative_error
                <
                0.5

                and

                AC_cosine
                >
                0.9
            )


            shot_noise_rows.append({

                "measurement_budget":
                    measurement_budget,

                "n_shots":
                    n_shots,

                "replicate":
                    replicate,

                "M0_success":
                    M0_noise_fit[
                        "success"
                    ],

                "M1_success":
                    M1_noise_fit[
                        "success"
                    ],

                "M0_nfev":
                    M0_noise_fit[
                        "nfev"
                    ],

                "M1_nfev":
                    M1_noise_fit[
                        "nfev"
                    ],

                "M0_test_rms":
                    M0_noise_fit[
                        "test_rms"
                    ],

                "M1_test_rms":
                    M1_noise_fit[
                        "test_rms"
                    ],

                "heldout_improvement_factor":
                    heldout_improvement,

                "learned_AC_norm":
                    np.linalg.norm(
                        learned_AC_noise
                    ),

                "AC_relative_error":
                    AC_relative_error,

                "AC_cosine":
                    AC_cosine,

                "AC_amplitude":
                    AC_amplitude,

                "AC_direction_residual":
                    AC_direction_residual,

                "phantom_recovered":
                    phantom_recovered
            })


shot_noise_results = pd.DataFrame(
    shot_noise_rows
)

In [38]:
# ------------------------------------------------------------------
# 4. Bootstrap summaries
# ------------------------------------------------------------------

def quantile_05(values):
    return np.quantile(
        values,
        0.05
    )


def quantile_95(values):
    return np.quantile(
        values,
        0.95
    )


shot_noise_summary = (

    shot_noise_results

    .groupby(
        [
            "measurement_budget",
            "n_shots"
        ]
    )

    .agg(

        number_of_replicates=(
            "replicate",
            "count"
        ),

        M0_success_rate=(
            "M0_success",
            "mean"
        ),

        M1_success_rate=(
            "M1_success",
            "mean"
        ),

        recovery_rate=(
            "phantom_recovered",
            "mean"
        ),

        median_M0_test_rms=(
            "M0_test_rms",
            "median"
        ),

        median_M1_test_rms=(
            "M1_test_rms",
            "median"
        ),

        median_improvement=(
            "heldout_improvement_factor",
            "median"
        ),

        median_AC_relative_error=(
            "AC_relative_error",
            "median"
        ),

        AC_relative_error_q05=(
            "AC_relative_error",
            quantile_05
        ),

        AC_relative_error_q95=(
            "AC_relative_error",
            quantile_95
        ),

        median_AC_cosine=(
            "AC_cosine",
            "median"
        ),

        AC_cosine_q05=(
            "AC_cosine",
            quantile_05
        ),

        median_AC_norm=(
            "learned_AC_norm",
            "median"
        )
    )

    .reset_index()
)


display(
    shot_noise_summary
)

,measurement_budget,n_shots,number_of_replicates,M0_success_rate,M1_success_rate,recovery_rate,median_M0_test_rms,median_M1_test_rms,median_improvement,median_AC_relative_error,AC_relative_error_q05,AC_relative_error_q95,median_AC_cosine,AC_cosine_q05,median_AC_norm
0,27,10000,20,1.0,1.0,0.00,0.007441,0.009001,0.818244,14.267500,9.982794,20.526860,0.085215,-0.446368,0.394216
1,27,100000,20,1.0,1.0,0.00,0.002254,0.002874,0.838802,4.333053,2.786226,5.907435,0.336907,-0.304690,0.122963
2,27,1000000,20,1.0,1.0,0.05,0.000812,0.000896,0.896809,1.490048,0.799289,1.942178,0.656679,0.347926,0.049888
3,54,10000,20,1.0,1.0,0.00,0.006392,0.008063,0.832885,11.679476,6.906499,15.903731,0.190107,-0.304856,0.325797
4,54,100000,20,1.0,1.0,0.00,0.002268,0.002821,0.819865,4.499616,2.423067,6.155583,0.314957,-0.064843,0.129050
5,54,1000000,20,1.0,1.0,0.00,0.000796,0.000832,0.928502,1.279823,0.938055,1.845520,0.634635,0.106973,0.041041
6,108,10000,20,1.0,1.0,0.00,0.005751,0.007201,0.819755,10.653571,8.185975,18.118775,0.097162,-0.435975,0.292909
7,108,100000,20,1.0,1.0,0.00,0.001691,0.002450,0.795825,3.192851,2.627564,5.504935,0.389613,-0.075066,0.093037
8,108,1000000,20,1.0,1.0,0.00,0.000691,0.000715,0.940866,1.193938,0.845165,1.747534,0.676562,0.361003,0.043444
9,216,10000,20,1.0,1.0,0.00,0.004059,0.005404,0.765973,8.496016,5.872341,12.766687,0.201897,-0.306489,0.236184


In [40]:
# ------------------------------------------------------------------
# Revised finite-shot diagnostics
#
# Optimization success and statistical recovery are different.
# A failed recovery is a valid scientific result, not a code failure.
# ------------------------------------------------------------------

noiseless_phantom_observable_scale = M0_test_rms

maximum_shot_count = int(
    shot_counts.max()
)

worst_case_single_setting_noise = (
    1.0
    /
    np.sqrt(
        maximum_shot_count
    )
)

single_setting_snr_proxy = (

    noiseless_phantom_observable_scale

    /

    worst_case_single_setting_noise
)


# These are only order-of-magnitude single-setting estimates.
# Measurement redundancy and conditioning also affect the true threshold.

estimated_shots_for_snr_1 = (
    1.0
    /
    noiseless_phantom_observable_scale
) ** 2

estimated_shots_for_snr_3 = (
    3.0
    /
    noiseless_phantom_observable_scale
) ** 2

estimated_shots_for_snr_5 = (
    5.0
    /
    noiseless_phantom_observable_scale
) ** 2


high_resource_detectability_pass = (

    high_resource_row[
        "recovery_rate"
    ]
    >= 0.80

    and

    high_resource_row[
        "median_M1_test_rms"
    ]
    <
    high_resource_row[
        "median_M0_test_rms"
    ]

    and

    high_resource_row[
        "median_AC_relative_error"
    ]
    <
    0.50

    and

    high_resource_row[
        "median_AC_cosine"
    ]
    >
    0.90
)


finite_shot_diagnostic_summary = pd.DataFrame({

    "quantity": [
        "noiseless M0 test residual",
        "worst-case single-setting noise at maximum shots",
        "single-setting SNR proxy",
        "estimated shots for SNR 1",
        "estimated shots for SNR 3",
        "estimated shots for SNR 5",
        "high-resource recovery rate",
        "high-resource median M0 test RMS",
        "high-resource median M1 test RMS",
        "high-resource median AC relative error",
        "high-resource median AC cosine",
        "high-resource detectability passed"
    ],

    "value": [
        noiseless_phantom_observable_scale,
        worst_case_single_setting_noise,
        single_setting_snr_proxy,
        estimated_shots_for_snr_1,
        estimated_shots_for_snr_3,
        estimated_shots_for_snr_5,
        high_resource_row[
            "recovery_rate"
        ],
        high_resource_row[
            "median_M0_test_rms"
        ],
        high_resource_row[
            "median_M1_test_rms"
        ],
        high_resource_row[
            "median_AC_relative_error"
        ],
        high_resource_row[
            "median_AC_cosine"
        ],
        high_resource_detectability_pass
    ]
})


display(
    finite_shot_diagnostic_summary
)


# Numerical integrity checks remain hard assertions.

assert np.isfinite(
    shot_noise_results.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()


assert (
    shot_noise_results[
        "M0_success"
    ].mean()
    >
    0.95
)


assert (
    shot_noise_results[
        "M1_success"
    ].mean()
    >
    0.95
)


print(
    "Finite-shot bootstrap numerical integrity: PASS"
)

print(
    "Optimization convergence: PASS"
)

print()


if high_resource_detectability_pass:

    print(
        "Finite-shot phantom recovery within current resource grid: PASS"
    )

else:

    print(
        "Finite-shot phantom recovery within current resource grid: "
        "NOT ACHIEVED"
    )

    print(
        "This is a statistical detectability result, not a code failure."
    )


print()

print(
    "Single-setting SNR proxy at maximum shots:",
    f"{single_setting_snr_proxy:.3f}"
)

print(
    "Estimated single-setting shots for SNR 3:",
    f"{estimated_shots_for_snr_3:.3e}"
)

print(
    "Estimated single-setting shots for SNR 5:",
    f"{estimated_shots_for_snr_5:.3e}"
)

,quantity,value
0,noiseless M0 test residual,0.000367
1,worst-case single-setting noise at maximum shots,0.001
2,single-setting SNR proxy,0.366842
3,estimated shots for SNR 1,7430902.088259
4,estimated shots for SNR 3,66878118.794332
5,estimated shots for SNR 5,185772552.206478
6,high-resource recovery rate,0.1
7,high-resource median M0 test RMS,0.000507
8,high-resource median M1 test RMS,0.000519
9,high-resource median AC relative error,0.926326


Finite-shot bootstrap numerical integrity: PASS
Optimization convergence: PASS

Finite-shot phantom recovery within current resource grid: NOT ACHIEVED
This is a statistical detectability result, not a code failure.

Single-setting SNR proxy at maximum shots: 0.367
Estimated single-setting shots for SNR 3: 6.688e+07
Estimated single-setting shots for SNR 5: 1.858e+08


In [41]:
shot_noise_results.to_csv(
    stage3_result_dir
    / "learner_shot_noise_bootstrap_raw.csv",
    index=False
)


shot_noise_summary.to_csv(
    stage3_result_dir
    / "learner_shot_noise_bootstrap_summary.csv",
    index=False
)


shot_noise_integrity_summary.to_csv(
    stage3_result_dir
    / "learner_shot_noise_integrity_summary.csv",
    index=False
)


finite_shot_diagnostic_summary.to_csv(
    stage3_result_dir
    / "learner_shot_noise_detectability_diagnostic.csv",
    index=False
)

ϵ-shots detectability phase diagram

In [42]:
# ------------------------------------------------------------------
# Epsilon-shots detectability phase diagram
#
# Fixed compressed measurement budget:
#     B = 216 settings
#
# Expected signal law:
#     observable phantom signal ~ epsilon^3
#
# Expected shot requirement at fixed SNR:
#     N_shots ~ epsilon^(-6)
# ------------------------------------------------------------------

tradeoff_seed = 20260811
tradeoff_rng = np.random.default_rng(
    tradeoff_seed
)


tradeoff_measurement_budget = 216


tradeoff_epsilon_values = np.array([
    0.08,
    0.12,
    0.16
])


tradeoff_shot_counts = np.array([
    100_000,
    300_000,
    1_000_000,
    3_000_000,
    10_000_000
])


n_tradeoff_replicates = 12


# ==================================================================
# 1. Freeze the compressed measurement design
# ==================================================================

tradeoff_flat_indices = (
    measurement_selection_order[
        :tradeoff_measurement_budget
    ]
)


tradeoff_state_indices = (
    tradeoff_flat_indices
    //
    n_observables
)


tradeoff_observable_indices = (
    tradeoff_flat_indices
    %
    n_observables
)


assert (
    len(
        tradeoff_flat_indices
    )
    ==
    tradeoff_measurement_budget
)


tradeoff_rows = []


# ==================================================================
# 2. Loop over protocol strength
# ==================================================================

for epsilon in tradeoff_epsilon_values:

    # --------------------------------------------------------------
    # Oracle data generation only
    # --------------------------------------------------------------

    protocol_hams, protocol_areas = cf.strang_lists(
        oracle_H_AB,
        oracle_H_BC,
        epsilon
    )


    oracle_U_epsilon = cf.protocol_unitary(
        protocol_hams,
        protocol_areas
    )


    branch_distance, _ = cf.unitary_branch_distance(
        oracle_U_epsilon
    )


    output_states_epsilon = (
        oracle_U_epsilon
        @ learner_input_states.T
    ).T


    output_expectations_epsilon = batched_expectations(
        output_states_epsilon,
        learner_observables
    )


    train_targets_epsilon = (
        output_expectations_epsilon[
            train_mask
        ]
    )


    test_targets_epsilon = (
        output_expectations_epsilon[
            test_mask
        ]
    )


    train_targets_flat_epsilon = (
        train_targets_epsilon.reshape(-1)
    )


    exact_selected_targets = (
        train_targets_flat_epsilon[
            tradeoff_flat_indices
        ]
    )


    exact_selected_inputs = (
        full_train_input_flat[
            tradeoff_flat_indices
        ]
    )


    # --------------------------------------------------------------
    # Epsilon-specific learner-visible linear sensitivity
    # --------------------------------------------------------------

    sensitivity_epsilon = (

        full_sensitivity_matrix

        *

        epsilon
        / learner_epsilon
    )


    selected_sensitivity_epsilon = (
        sensitivity_epsilon[
            tradeoff_flat_indices
        ]
    )


    # --------------------------------------------------------------
    # Exact finite-step AC target for post-fit evaluation only
    # --------------------------------------------------------------

    oracle_L_epsilon = cf.log_unitary(
        oracle_U_epsilon
    )


    oracle_G_epsilon = (
        1j
        * oracle_L_epsilon
        / epsilon
    )


    oracle_AC_epsilon = cf.exact_support_component(
        oracle_G_epsilon,
        {A, C},
        local_dim=2,
        n_sites=N
    )


    oracle_AC_norm_epsilon = np.linalg.norm(
        oracle_AC_epsilon
    )


    # --------------------------------------------------------------
    # Finite-shot realizations
    # --------------------------------------------------------------

    for n_shots in tradeoff_shot_counts:

        for replicate in range(
            n_tradeoff_replicates
        ):

            noisy_selected_targets = (
                sample_pauli_expectations(
                    exact_selected_targets,
                    n_shots,
                    tradeoff_rng
                )
            )


            noisy_linear_target = (
                noisy_selected_targets
                -
                exact_selected_inputs
            )


            fitted_tradeoff_models = {}


            for model_name, model in model_definitions.items():

                parameter_indices = model[
                    "parameter_indices"
                ]


                basis = model[
                    "basis_matrices"
                ]


                model_sensitivity = (
                    selected_sensitivity_epsilon[
                        :,
                        parameter_indices
                    ]
                )


                theta_linear, *_ = np.linalg.lstsq(
                    model_sensitivity,
                    noisy_linear_target,
                    rcond=None
                )


                initialization_candidates = [
                    theta_linear,
                    np.zeros_like(
                        theta_linear
                    )
                ]


                initialization_errors = []


                for candidate in initialization_candidates:

                    candidate_residual = noisy_selected_residual(

                        candidate,
                        basis,
                        train_states,
                        epsilon,
                        tradeoff_state_indices,
                        tradeoff_observable_indices,
                        noisy_selected_targets
                    )


                    initialization_errors.append(

                        np.sqrt(
                            np.mean(
                                candidate_residual**2
                            )
                        )
                    )


                theta_initial = (
                    initialization_candidates[
                        int(
                            np.argmin(
                                initialization_errors
                            )
                        )
                    ]
                )


                fit_result = least_squares(

                    noisy_selected_residual,

                    x0=
                        theta_initial,

                    args=(
                        basis,
                        train_states,
                        epsilon,
                        tradeoff_state_indices,
                        tradeoff_observable_indices,
                        noisy_selected_targets
                    ),

                    method=
                        "trf",

                    jac=
                        "2-point",

                    x_scale=
                        "jac",

                    ftol=
                        1e-11,

                    xtol=
                        1e-11,

                    gtol=
                        1e-11,

                    max_nfev=
                        400,

                    verbose=
                        0
                )


                test_metrics = prediction_metrics(
                    fit_result.x,
                    basis,
                    test_states,
                    test_targets_epsilon,
                    epsilon
                )


                fitted_tradeoff_models[
                    model_name
                ] = {

                    "parameters":
                        fit_result.x.copy(),

                    "success":
                        fit_result.success,

                    "nfev":
                        fit_result.nfev,

                    "test_rms":
                        test_metrics[
                            "rms"
                        ]
                }


            # ------------------------------------------------------
            # Extract inferred AC
            # ------------------------------------------------------

            M0_tradeoff_fit = fitted_tradeoff_models[
                "M0_micro_edges_only"
            ]


            M1_tradeoff_fit = fitted_tradeoff_models[
                "M1_with_candidate_AC"
            ]


            M1_tradeoff_parameters = (
                M1_tradeoff_fit[
                    "parameters"
                ]
            )


            learned_AC_tradeoff = generator_from_parameters(

                M1_tradeoff_parameters[
                    M1_AC_mask
                ],

                model_definitions[
                    "M1_with_candidate_AC"
                ][
                    "basis_matrices"
                ][
                    M1_AC_mask
                ]
            )


            AC_relative_error = relative_frobenius_error(
                learned_AC_tradeoff,
                oracle_AC_epsilon
            )


            (
                AC_cosine,
                AC_amplitude,
                AC_direction_residual
            ) = hilbert_schmidt_alignment(
                learned_AC_tradeoff,
                oracle_AC_epsilon
            )


            heldout_improvement = (

                M0_tradeoff_fit[
                    "test_rms"
                ]

                /

                max(
                    M1_tradeoff_fit[
                        "test_rms"
                    ],
                    1e-300
                )
            )


            phantom_recovered = (

                M1_tradeoff_fit[
                    "test_rms"
                ]
                <
                M0_tradeoff_fit[
                    "test_rms"
                ]

                and

                AC_relative_error
                <
                0.5

                and

                AC_cosine
                >
                0.9
            )


            tradeoff_rows.append({

                "epsilon":
                    epsilon,

                "n_shots":
                    n_shots,

                "measurement_budget":
                    tradeoff_measurement_budget,

                "replicate":
                    replicate,

                "branch_distance":
                    branch_distance,

                "M0_success":
                    M0_tradeoff_fit[
                        "success"
                    ],

                "M1_success":
                    M1_tradeoff_fit[
                        "success"
                    ],

                "M0_test_rms":
                    M0_tradeoff_fit[
                        "test_rms"
                    ],

                "M1_test_rms":
                    M1_tradeoff_fit[
                        "test_rms"
                    ],

                "heldout_improvement_factor":
                    heldout_improvement,

                "oracle_AC_norm":
                    oracle_AC_norm_epsilon,

                "learned_AC_norm":
                    np.linalg.norm(
                        learned_AC_tradeoff
                    ),

                "AC_relative_error":
                    AC_relative_error,

                "AC_cosine":
                    AC_cosine,

                "AC_amplitude":
                    AC_amplitude,

                "AC_direction_residual":
                    AC_direction_residual,

                "phantom_recovered":
                    phantom_recovered
            })


epsilon_shot_results = pd.DataFrame(
    tradeoff_rows
)


# ==================================================================
# 3. Aggregate the detectability map
# ==================================================================

def q10(values):
    return np.quantile(
        values,
        0.10
    )


def q90(values):
    return np.quantile(
        values,
        0.90
    )


epsilon_shot_summary = (

    epsilon_shot_results

    .groupby(
        [
            "epsilon",
            "n_shots"
        ]
    )

    .agg(

        number_of_replicates=(
            "replicate",
            "count"
        ),

        M0_success_rate=(
            "M0_success",
            "mean"
        ),

        M1_success_rate=(
            "M1_success",
            "mean"
        ),

        recovery_rate=(
            "phantom_recovered",
            "mean"
        ),

        median_M0_test_rms=(
            "M0_test_rms",
            "median"
        ),

        median_M1_test_rms=(
            "M1_test_rms",
            "median"
        ),

        median_improvement=(
            "heldout_improvement_factor",
            "median"
        ),

        median_AC_relative_error=(
            "AC_relative_error",
            "median"
        ),

        AC_relative_error_q10=(
            "AC_relative_error",
            q10
        ),

        AC_relative_error_q90=(
            "AC_relative_error",
            q90
        ),

        median_AC_cosine=(
            "AC_cosine",
            "median"
        ),

        AC_cosine_q10=(
            "AC_cosine",
            q10
        ),

        median_learned_AC_norm=(
            "learned_AC_norm",
            "median"
        ),

        oracle_AC_norm=(
            "oracle_AC_norm",
            "first"
        )
    )

    .reset_index()
)


display(
    epsilon_shot_summary
)

,epsilon,n_shots,number_of_replicates,M0_success_rate,M1_success_rate,recovery_rate,median_M0_test_rms,median_M1_test_rms,median_improvement,median_AC_relative_error,AC_relative_error_q10,AC_relative_error_q90,median_AC_cosine,AC_cosine_q10,median_learned_AC_norm,oracle_AC_norm
0,0.08,100000,12,1.0,1.0,0.000000,0.001314,0.001702,0.775631,3.013111,2.271341,3.567177,0.357232,0.037461,0.086779,0.027473
1,0.08,300000,12,1.0,1.0,0.000000,0.000855,0.001051,0.792475,1.675726,1.115855,2.309833,0.605234,0.046675,0.050838,0.027473
2,0.08,1000000,12,1.0,1.0,0.000000,0.000562,0.000563,1.020471,0.988634,0.634321,1.215245,0.769377,0.560629,0.040768,0.027473
3,0.08,3000000,12,1.0,1.0,0.416667,0.000437,0.000290,1.540960,0.508927,0.324641,0.624416,0.912324,0.833786,0.028286,0.027473
4,0.08,10000000,12,1.0,1.0,1.000000,0.000417,0.000173,2.367269,0.243606,0.159727,0.390603,0.980077,0.963337,0.028502,0.027473
5,0.12,100000,12,1.0,1.0,0.000000,0.001682,0.001618,1.061581,0.833513,0.681590,0.972188,0.814605,0.697650,0.082091,0.062264
6,0.12,300000,12,1.0,1.0,0.750000,0.001429,0.000796,1.809236,0.389180,0.296157,0.488995,0.948950,0.876991,0.061568,0.062264
7,0.12,1000000,12,1.0,1.0,1.000000,0.001370,0.000509,2.633926,0.243678,0.158192,0.333184,0.976379,0.943970,0.061856,0.062264
8,0.12,3000000,12,1.0,1.0,1.000000,0.001310,0.000288,4.507872,0.131367,0.105722,0.220364,0.992588,0.977417,0.062919,0.062264
9,0.12,10000000,12,1.0,1.0,1.000000,0.001298,0.000165,7.828375,0.083411,0.072902,0.102928,0.997088,0.996700,0.061644,0.062264


In [43]:
# ==================================================================
# 4. Recovery-rate phase table
# ==================================================================

recovery_rate_table = (

    epsilon_shot_summary

    .pivot(
        index=
            "epsilon",

        columns=
            "n_shots",

        values=
            "recovery_rate"
    )
)


display(
    recovery_rate_table
)


# Stable recovery criterion:
#
#   at least 80% of independent shot-noise realizations recover AC.

epsilon_shot_summary[
    "stable_recovery"
] = (

    epsilon_shot_summary[
        "recovery_rate"
    ]
    >= 0.80
)


stable_rows = epsilon_shot_summary.loc[
    epsilon_shot_summary[
        "stable_recovery"
    ]
]


threshold_rows = []


for epsilon in tradeoff_epsilon_values:

    epsilon_rows = (
        epsilon_shot_summary

        .loc[
            epsilon_shot_summary[
                "epsilon"
            ]
            ==
            epsilon
        ]

        .sort_values(
            "n_shots"
        )
    )


    stable_epsilon_rows = epsilon_rows.loc[
        epsilon_rows[
            "stable_recovery"
        ]
    ]


    if len(
        stable_epsilon_rows
    ) > 0:

        threshold_shots = int(

            stable_epsilon_rows[
                "n_shots"
            ].iloc[0]
        )

    else:

        threshold_shots = np.nan


    threshold_rows.append({

        "epsilon":
            epsilon,

        "minimum_tested_shots_for_80pct_recovery":
            threshold_shots,

        "epsilon_to_minus6":
            epsilon**(-6)
    })


epsilon_shot_thresholds = pd.DataFrame(
    threshold_rows
)


display(
    epsilon_shot_thresholds
)

n_shots,100000,300000,1000000,3000000,10000000
epsilon,,,,,
0.08,0.000000,0.00,0.0,0.416667,1.0
0.12,0.000000,0.75,1.0,1.000000,1.0
0.16,0.916667,1.00,1.0,1.000000,1.0


,epsilon,minimum_tested_shots_for_80pct_recovery,epsilon_to_minus6
0,0.08,10000000,3.814697e+06
1,0.12,1000000,3.348980e+05
2,0.16,100000,5.960464e+04


In [44]:
# ==================================================================
# 5. Integrity checks
# ==================================================================

assert np.isfinite(
    epsilon_shot_results.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()


assert (
    epsilon_shot_results[
        "M0_success"
    ].mean()
    >
    0.95
)


assert (
    epsilon_shot_results[
        "M1_success"
    ].mean()
    >
    0.95
)


assert (
    epsilon_shot_results[
        "branch_distance"
    ].min()
    >
    1.0
)


print(
    "Epsilon-shots phase diagram numerical integrity: PASS"
)

print(
    "Optimization convergence: PASS"
)

print(
    "All protocols remain branch-safe: PASS"
)

print()


if len(
    stable_rows
) > 0:

    best_stable_row = (

        stable_rows

        .sort_values(
            [
                "n_shots",
                "epsilon"
            ],

            ascending=[
                True,
                False
            ]
        )

        .iloc[0]
    )


    print(
        "Stable finite-shot recovery found within tested grid: YES"
    )

    print(
        "Lowest tested stable shot count:",
        int(
            best_stable_row[
                "n_shots"
            ]
        )
    )

    print(
        "Corresponding epsilon:",
        f"{best_stable_row['epsilon']:.3f}"
    )

    print(
        "Recovery rate:",
        f"{best_stable_row['recovery_rate']:.3f}"
    )

    print(
        "Median AC relative error:",
        f"{best_stable_row['median_AC_relative_error']:.3f}"
    )

else:

    print(
        "Stable finite-shot recovery found within tested grid: NO"
    )

    print(
        "The detectability boundary lies beyond the current grid."
    )


epsilon_shot_results.to_csv(
    stage3_result_dir
    / "learner_epsilon_shot_tradeoff_raw.csv",
    index=False
)


epsilon_shot_summary.to_csv(
    stage3_result_dir
    / "learner_epsilon_shot_tradeoff_summary.csv",
    index=False
)


epsilon_shot_thresholds.to_csv(
    stage3_result_dir
    / "learner_epsilon_shot_thresholds.csv",
    index=False
)


np.savez(
    stage3_result_dir
    / "learner_epsilon_shot_tradeoff_settings.npz",

    epsilon_values=
        tradeoff_epsilon_values,

    shot_counts=
        tradeoff_shot_counts,

    measurement_budget=
        tradeoff_measurement_budget,

    n_replicates=
        n_tradeoff_replicates,

    seed=
        tradeoff_seed
)

Epsilon-shots phase diagram numerical integrity: PASS
Optimization convergence: PASS
All protocols remain branch-safe: PASS

Stable finite-shot recovery found within tested grid: YES
Lowest tested stable shot count: 100000
Corresponding epsilon: 0.160
Recovery rate: 0.917
Median AC relative error: 0.295
